In [ ]:
import os
import datetime
import logging
import random
import gc  # [추가] 가비지 컬렉터 임포트

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast # [수정] autocast 명시적 임포트

from netCDF4 import Dataset as ncDataset
from tqdm import tqdm
import optuna
from sklearn.metrics import mean_squared_error, mean_absolute_error


In [ ]:
import os
import random
import datetime
import logging
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
from netCDF4 import Dataset as ncDataset
from tqdm import tqdm
from sklearn.metrics import mean_squared_error, mean_absolute_error
import rasterio
from rasterio.enums import Resampling

# ─────────────────────────────────────────────────────────
# 0) 전역 설정
# ─────────────────────────────────────────────────────────
NUM_WORKERS = 2
PIN_MEMORY = True

# 제주 영역: 0.01° 해상도 (목표 격자)
JEJU_LAT_START, JEJU_LAT_END = 33.1, 33.7  # 61 셀
JEJU_LON_START, JEJU_LON_END = 126.1, 127.1
RESOLUTION = 0.01
LAT_VALS = np.arange(JEJU_LAT_START, JEJU_LAT_END + RESOLUTION/2, RESOLUTION, dtype='f4')
LON_VALS = np.arange(JEJU_LON_START, JEJU_LON_END + RESOLUTION/2, RESOLUTION, dtype='f4')

# ─────────────────────────────────────────────────────────
# 1) GeoTIFF DEM 로딩 함수
# ─────────────────────────────────────────────────────────
def load_dem_geotiff(dem_path, target_shape):
    """
    GeoTIFF DEM 파일을 bilinear 리샘플링하여 target_shape (height, width) 로 반환
    """
    with rasterio.open(dem_path) as src:
        data = src.read(
            1,
            out_shape=target_shape,
            resampling=Resampling.bilinear
        )
    return data.astype(np.float32)

# ─────────────────────────────────────────────────────────
# 2) NetCDF 저장 함수 (위·경도 좌표 포함)
# ─────────────────────────────────────────────────────────
def save_to_netcdf(data, date_str, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    outfn = os.path.join(output_dir, f"{date_str}_Downscaled_Precipitation.nc")
    n_lat, n_lon = LAT_VALS.size, LON_VALS.size
    time_unit = f"days since {date_str[:4]}-{date_str[4:6]}-{date_str[6:]} 00:00:00"
    with ncDataset(outfn, 'w', format='NETCDF4') as ds:
        ds.createDimension('time', 1)
        ds.createDimension('lat', n_lat)
        ds.createDimension('lon', n_lon)
        tv = ds.createVariable('time', 'f4', ('time',))
        tv[:] = [0.0]; tv.units = time_unit; tv.calendar = 'gregorian'
        lv = ds.createVariable('lat', 'f4', ('lat',))
        lv[:] = LAT_VALS; lv.units = 'degrees_north'; lv.standard_name = 'latitude'; lv.axis = 'Y'
        lov = ds.createVariable('lon', 'f4', ('lon',))
        lov[:] = LON_VALS; lov.units = 'degrees_east'; lov.standard_name = 'longitude'; lov.axis = 'X'
        p = ds.createVariable('precipitation','f4',('time','lat','lon'),
                              zlib=True, complevel=4, fill_value=-9999.0)
        p[0,:,:] = data.astype('f4')
        p.units = 'mm/day'; p.long_name = 'Downscaled Precipitation'
        p.coordinates = 'time lat lon'; p.grid_mapping = 'crs'
        crs = ds.createVariable('crs','i4')
        crs.grid_mapping_name = 'latitude_longitude'; crs.epsg_code = 'EPSG:4326'
        crs.semi_major_axis = 6378137.0; crs.inverse_flattening = 298.257223563
        ds.Conventions = 'CF-1.8'
        ds.title = f"Downscaled Precipitation for {date_str}"
        ds.history = f"Created on {datetime.datetime.now():%Y-%m-%d %H:%M:%S}"  

# ─────────────────────────────────────────────────────────
# 3) Dataset 클래스 (GeoTIFF DEM 사용)
# ─────────────────────────────────────────────────────────
class RainfallDataset(Dataset):
    def __init__(self, low_res_folder, high_res_folder, dem_file,
                 start_date, end_date, normalization_stats=None):
        self.low = low_res_folder
        self.high = high_res_folder
        self.dates = [start_date + datetime.timedelta(days=i)
                      for i in range((end_date - start_date).days + 1)]
        self.paths = []
        for d in self.dates:
            f1 = os.path.join(self.low, d.strftime('%Y%m%d') + '.nc4')
            f2 = os.path.join(self.high, d.strftime('%Y%m%d') + '_Cokriging_Precipitation.nc')
            if os.path.exists(f1) and os.path.exists(f2):
                self.paths.append((f1,f2))
        if not self.paths:
            raise ValueError("유효한 데이터가 없습니다.")
        print(f"유효한 데이터 쌍: {len(self.paths)}개")
        print("DEM GeoTIFF 로딩 및 재샘플링 중...")
        self.dem = load_dem_geotiff(
            dem_file,
            target_shape=(LAT_VALS.size, LON_VALS.size)
        )
        print(f"DEM 최종 크기: {self.dem.shape}")
        self.stats = normalization_stats or self._calc_stats()
    def __len__(self): return len(self.paths)
    def __getitem__(self, idx):
        low_f, high_f = self.paths[idx]
        low_arr = self._load_nc(low_f, 'precipitation')
        # 리샘플 if needed
        if low_arr.shape != (LAT_VALS.size, LON_VALS.size):
            t = torch.tensor(low_arr, dtype=torch.float32).unsqueeze(0).unsqueeze(0)
            low_arr = torch.nn.functional.interpolate(
                t, size=(LAT_VALS.size, LON_VALS.size),
                mode='bilinear', align_corners=False
            ).squeeze().numpy()
        high_arr = self._load_nc(high_f, 'precipitation')
        # 정규화
        low_norm = (low_arr - self.stats['low_res']['mean']) / (self.stats['low_res']['std'] + 1e-8)
        high_norm= (high_arr- self.stats['high_res']['mean'])/ (self.stats['high_res']['std']+1e-8)
        dem_norm = (self.dem - self.stats['dem']['mean']) / (self.stats['dem']['std']+1e-8)
        x = torch.tensor(np.stack([low_norm, dem_norm], axis=0), dtype=torch.float32)
        y = torch.tensor(high_norm, dtype=torch.float32).unsqueeze(0)
        return x, y
    def _load_nc(self, path, var):
        with ncDataset(path) as nc:
            arr = nc.variables[var][:].astype(np.float32)
            return arr.squeeze() if arr.ndim==3 else arr
    def _calc_stats(self):
        print("정규화 통계 계산 중...")
        lows, highs = [], []
        for i, (l, h) in enumerate(self.paths):
            if i % 100 == 0: print(f"통계[{i}/{len(self.paths)}]")
            lows.append(load_dem_geotiff(l, (LAT_VALS.size, LON_VALS.size)) if False else self._load_nc(l,'precipitation'))
            highs.append(self._load_nc(h,'precipitation'))
        lows  = np.stack([
            torch.nn.functional.interpolate(
                torch.tensor(a, dtype=torch.float32).unsqueeze(0).unsqueeze(0),
                size=(LAT_VALS.size,LON_VALS.size), mode='bilinear', align_corners=False
            ).squeeze().numpy() if a.shape!=(LAT_VALS.size,LON_VALS.size) else a
            for a in lows
        ])
        highs = np.stack(highs)
        stats = {
            'low_res':  {'mean': lows.mean(),  'std': lows.std()},
            'high_res': {'mean': highs.mean(), 'std': highs.std()},
            'dem':      {'mean': self.dem.mean(), 'std': self.dem.std()}
        }
        print("통계 완료:")
        for k,v in stats.items(): print(f" {k}: mean={v['mean']:.4f}, std={v['std']:.4f}")
        return stats

# ─────────────────────────────────────────────────────────
# 4) Generator 모델 정의
# ─────────────────────────────────────────────────────────
class Generator(nn.Module):
    def __init__(self, in_ch=2, n_layers=4, base_filters=32):
        super().__init__()
        layers=[]; c=in_ch
        for i in range(n_layers):
            f=base_filters*(2**i)
            layers += [nn.Conv2d(c,f,3,1,1,bias=False), nn.BatchNorm2d(f), nn.ReLU(True)]
            c=f
        self.encoder=nn.Sequential(*layers)
        self.final  =nn.Conv2d(c,1,3,1,1)
    def forward(self,x): return self.final(self.encoder(x))

# ─────────────────────────────────────────────────────────
# 5) 학습/테스트 함수 (하이퍼옵션 생략)
# ─────────────────────────────────────────────────────────
def train_final(train_ds, val_ds, test_ds, params, log_dir, epochs=50):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    logging.info(f"Using device: {device}")
    G=Generator(2, params['n_layers_g'], params['n_filters_base_g']).to(device)
    optimG=optim.AdamW(G.parameters(), lr=params['lr_g'], betas=(0.5,0.999))
    criterion=nn.L1Loss() if params['loss_function']=='L1' else nn.MSELoss()
    scaler=GradScaler(enabled=torch.cuda.is_available())
    tl=DataLoader(train_ds, batch_size=params['batch_size'], shuffle=True, pin_memory=PIN_MEMORY, num_workers=NUM_WORKERS)
    vl=DataLoader(val_ds,   batch_size=params['batch_size'], shuffle=False,pin_memory=PIN_MEMORY, num_workers=NUM_WORKERS)
    best, cnt=1e9,0
    for ep in range(1,epochs+1):
        G.train(); train_loss=0
        for x,y in tl:
            x,y=x.to(device),y.to(device)
            optimG.zero_grad()
            with autocast(enabled=torch.cuda.is_available()): loss=criterion(G(x),y)*params['content_weight']
            scaler.scale(loss).backward(); scaler.step(optimG); scaler.update()
            train_loss+=loss.item()
        train_loss/=len(tl)
        G.eval(); val_loss=0
        with torch.no_grad():
            for x,y in vl: val_loss+=criterion(G(x.to(device)),y.to(device)).item()
        val_loss/=len(vl)
        logging.info(f"Epoch {ep}, Train {train_loss:.6f}, Val {val_loss:.6f}")
        if val_loss<best: best,val_cnt=val_loss,0; torch.save(G.state_dict(), os.path.join(log_dir,'best_generator.pth'))
        else: cnt+=1;
        if cnt>=15: break
    G.load_state_dict(torch.load(os.path.join(log_dir,'best_generator.pth')))
    # 테스트
    tle=DataLoader(test_ds, batch_size=1, shuffle=False)
    ms,ma=[],[]
    with torch.no_grad():
        for x,y in tle:
            x,y=x.to(device),y.to(device)
            p=G(x)
            y_,p_=y.cpu().numpy().flatten(),p.cpu().numpy().flatten()
            ms.append(mean_squared_error(y_,p_)); ma.append(mean_absolute_error(y_,p_))
    return G, np.mean(ms), np.mean(ma)

# ─────────────────────────────────────────────────────────
# 6) 추론 및 저장
# ─────────────────────────────────────────────────────────
def run_inference(model, train_ds, low_res_dir, output_dir, start_date, end_date):
    device=next(model.parameters()).device
    model.eval()
    dem_norm=(train_ds.dem - train_ds.stats['dem']['mean'])/(train_ds.stats['dem']['std']+1e-8)
    dates=[start_date+datetime.timedelta(days=i) for i in range((end_date-start_date).days+1)]
    for date in tqdm(dates, desc="Inference"):
        lf=os.path.join(low_res_dir,date.strftime('%Y%m%d')+'.nc4')
        if not os.path.exists(lf): continue
        with ncDataset(lf) as nc: arr=nc.variables['precipitation'][:].astype(np.float32)
        arr=arr.squeeze() if arr.ndim==3 else arr
        if arr.shape!=(LAT_VALS.size,LON_VALS.size):
            t=torch.tensor(arr,dtype=torch.float32).unsqueeze(0).unsqueeze(0)
            arr=torch.nn.functional.interpolate(t,size=(LAT_VALS.size,LON_VALS.size),mode='bilinear',align_corners=False).squeeze().numpy()
        low_norm=(arr-train_ds.stats['low_res']['mean'])/(train_ds.stats['low_res']['std']+1e-8)
        x=torch.tensor(np.stack([low_norm,dem_norm],axis=0),dtype=torch.float32).unsqueeze(0).to(device)
        with torch.no_grad(): pred=model(x).cpu().squeeze().numpy()
        pred_den=pred*train_ds.stats['high_res']['std']+train_ds.stats['high_res']['mean']
        pred_den=np.clip(pred_den,0,None)
        save_to_netcdf(pred_den,date.strftime('%Y%m%d'),output_dir)

# ─────────────────────────────────────────────────────────
# 7) 메인 실행
# ─────────────────────────────────────────────────────────
def main():
    random.seed(42); np.random.seed(42); torch.manual_seed(42)
    run_dt=datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
    log_dir=f"/mnt/c/Yeonji/2025.01.Drought/2004/{run_dt}"
    os.makedirs(log_dir, exist_ok=True)
    logging.basicConfig(level=logging.INFO,filename=os.path.join(log_dir,'train.log'),format='%(asctime)s - %(message)s')
    low_res_dir=r"/mnt/c/Yeonji/GPM/Input"
    high_res_dir=r"/mnt/c/Yeonji/2025.01.Drought/2004/1.CoKriging(Daily)"
    dem_file=r"/mnt/c/Yeonji/2025.01.Drought/DEM_jeju.tif"  # GeoTIFF
    print("데이터셋 로딩 중...")
    train_ds=RainfallDataset(low_res_dir,high_res_dir,dem_file,datetime.date(2004,1,1),datetime.date(2018,12,31))
    val_ds  =RainfallDataset(low_res_dir,high_res_dir,dem_file,datetime.date(2019,1,1),datetime.date(2021,12,31),normalization_stats=train_ds.stats)
    test_ds =RainfallDataset(low_res_dir,high_res_dir,dem_file,datetime.date(2022,1,1),datetime.date(2023,12,31),normalization_stats=train_ds.stats)
    print(f"Train/Val/Test sizes: {len(train_ds)}/{len(val_ds)}/{len(test_ds)}")
    best_params={'batch_size':32,'lr_g':1.6237e-05,'content_weight':137.9359,'n_layers_g':4,'n_filters_base_g':32,'loss_function':'L1'}
    print("모델 학습 시작...")
    model,mse,mae=train_final(train_ds,val_ds,test_ds,best_params,log_dir,epochs=50)
    print(f"▶ Test MSE: {mse:.4f}, MAE: {mae:.4f}")
    print("추론 및 저장 시작...")
    out_folder=r"/mnt/c/Yeonji/2025.01.Drought/2004/downscaled_results_final"
    run_inference(model,train_ds,low_res_dir,out_folder,datetime.date(2004,1,1),datetime.date(2023,12,31))
    print("✅ 완료!")

if __name__=='__main__':
    main()


In [ ]:
import os
import random
import datetime
import logging
import gc
import glob
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
from netCDF4 import Dataset as ncDataset
from tqdm import tqdm
import optuna
from sklearn.metrics import mean_squared_error, mean_absolute_error

# ======================================================
# 전역 설정 및 환경 변수
# ======================================================
NUM_WORKERS = 2
PIN_MEMORY = True
DEVICE = None

# ======================================================
# 0. 설정: 시드 고정, 디렉토리
# ======================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

def setup_directories(run_date):
    log_dir = f'/mnt/c/Yeonji/2025.01.Drought/2004/{run_date}'
    results_dir = os.path.join(log_dir, 'results')
    plots_dir = os.path.join(log_dir, 'plots')
    optuna_dir = os.path.join(log_dir, 'optuna')
    for d in [log_dir, results_dir, plots_dir, optuna_dir]:
        os.makedirs(d, exist_ok=True)
    return log_dir

def setup_logging(log_dir):
    logging.basicConfig(
        filename=os.path.join(log_dir, 'training_hyperopt.log'),
        level=logging.INFO,
        format='%(asctime)s - %(levelname)s - %(message)s'
    )

# ======================================================
# GPU 진단 및 설정
# ======================================================
def setup_gpu():
    print("="*50)
    print("=== 시스템 GPU 진단 ===")
    print(f"PyTorch 버전: {torch.__version__}")
    print(f"CUDA 사용 가능: {torch.cuda.is_available()}")

    if torch.cuda.is_available():
        os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'max_split_size_mb:512'
        print(f"CUDA 버전: {torch.version.cuda}")
        print(f"cuDNN 버전: {torch.backends.cudnn.version()}")
        print(f"GPU 개수: {torch.cuda.device_count()}")
        
        device = torch.device('cuda')
        print(f"✅ 최종 선택된 디바이스: {torch.cuda.get_device_name(0)} (cuda:0)")
    else:
        device = torch.device('cpu')
        print("❌ CUDA 사용 불가 - CPU 사용")
    
    print('='*50)
    return device

# ======================================================
# 데이터셋 정의
# ======================================================
class RainfallDataset(Dataset):
    def __init__(self, low_res_folder, high_res_folder, dem_file,
                 start_date, end_date, normalization_stats=None):
        self.low_res_folder = low_res_folder
        self.high_res_folder = high_res_folder
        self.dates = [start_date + datetime.timedelta(days=i)
                      for i in range((end_date - start_date).days + 1)]
        
        self.valid_data_paths = self._filter_valid_data()
        if not self.valid_data_paths:
            raise ValueError("데이터셋에 유효한 데이터가 없습니다.")

        self.dem_data = self._load_and_resize(dem_file, 'elevation', (61, 101))
        
        if normalization_stats is None:
            self.stats = self._calculate_normalization_stats_efficiently()
        else:
            self.stats = normalization_stats

    def __len__(self):
        return len(self.valid_data_paths)

    def __getitem__(self, idx):
        low_file, high_file = self.valid_data_paths[idx]
        
        low = self._load_and_resize(low_file, 'precipitation', (61, 101))
        high = self._load_nc(high_file, 'precipitation')
        
        # 정규화
        low = (low - self.stats['low_res']['mean']) / (self.stats['low_res']['std'] + 1e-8)
        high = (high - self.stats['high_res']['mean']) / (self.stats['high_res']['std'] + 1e-8)
        
        x = torch.from_numpy(np.stack([low, self.dem_data], axis=0)).float()
        y = torch.from_numpy(high).float().unsqueeze(0)
        
        return x, y

    def _filter_valid_data(self):
        valid_paths = []
        for d in self.dates:
            f1 = os.path.join(self.low_res_folder, d.strftime('%Y%m%d') + '.nc4')
            f2 = os.path.join(self.high_res_folder, d.strftime('%Y%m%d') + '_Cokriging_Precipitation.nc')
            if os.path.exists(f1) and os.path.exists(f2):
                valid_paths.append((f1, f2))
        logging.info(f"유효 날짜에 해당하는 파일 쌍: {len(valid_paths)} / 전체 기간: {len(self.dates)}일")
        return valid_paths

    def _load_nc(self, path, var):
        try:
            with ncDataset(path) as nc:
                arr = nc.variables[var][:].astype(np.float32)
                if arr.ndim == 3:
                    arr = arr.squeeze(0)
                return arr
        except Exception as e:
            logging.error(f"NC 로드 오류({path}): {e}")
            return np.zeros((61, 101), dtype=np.float32)

    def _load_and_resize(self, path, var, size):
        arr = self._load_nc(path, var)
        if arr.shape != size:
            t = torch.from_numpy(arr).float().unsqueeze(0).unsqueeze(0)
            resized = torch.nn.functional.interpolate(t, size=size, mode='bilinear', align_corners=False)
            return resized.squeeze().numpy()
        return arr

    def _calculate_normalization_stats_efficiently(self):
        stats = {
            'low_res': {'mean': 0.0, 'std': 0.0, 'm2': 0.0, 'count': 0},
            'high_res': {'mean': 0.0, 'std': 0.0, 'm2': 0.0, 'count': 0}
        }

        def update_stats(data, key):
            for x in np.nditer(data):
                stats[key]['count'] += 1
                delta = x - stats[key]['mean']
                stats[key]['mean'] += delta / stats[key]['count']
                delta2 = x - stats[key]['mean']
                stats[key]['m2'] += delta * delta2

        for low_path, high_path in tqdm(self.valid_data_paths, desc="Calculating Stats Efficiently"):
            low_data = self._load_and_resize(low_path, 'precipitation', (61, 101))
            high_data = self._load_nc(high_path, 'precipitation')
            update_stats(low_data, 'low_res')
            update_stats(high_data, 'high_res')

        for key in stats:
            if stats[key]['count'] > 1:
                stats[key]['std'] = np.sqrt(stats[key]['m2'] / stats[key]['count'])
            else:
                stats[key]['std'] = 0.0
        
        final_stats = {
            'low_res': {'mean': stats['low_res']['mean'], 'std': stats['low_res']['std']},
            'high_res': {'mean': stats['high_res']['mean'], 'std': stats['high_res']['std']}
        }
        logging.info(f"정규화 통계 계산 완료: {final_stats}")
        return final_stats

# ======================================================
# 추론용 데이터셋 클래스
# ======================================================
class InferenceRainfallDataset(Dataset):
    def __init__(self, low_res_folder, dem_file, start_date, end_date, normalization_stats):
        self.low_res_folder = low_res_folder
        self.dates = [start_date + datetime.timedelta(days=i)
                      for i in range((end_date - start_date).days + 1)]
        
        self.valid_data_info = self._filter_valid_data()
        if not self.valid_data_info:
            raise ValueError("추론할 유효한 데이터가 없습니다.")

        self.dem_data = self._load_and_resize(dem_file, 'elevation', (61, 101))
        self.stats = normalization_stats
        
        print(f"추론 대상: {len(self.valid_data_info)}개 파일")

    def __len__(self):
        return len(self.valid_data_info)

    def __getitem__(self, idx):
        date_str, low_file = self.valid_data_info[idx]
        
        low = self._load_and_resize(low_file, 'precipitation', (61, 101))
        low = (low - self.stats['low_res']['mean']) / (self.stats['low_res']['std'] + 1e-8)
        
        x = torch.from_numpy(np.stack([low, self.dem_data], axis=0)).float()
        
        return x, date_str

    def _filter_valid_data(self):
        valid_info = []
        for d in self.dates:
            date_str = d.strftime('%Y%m%d')
            low_file = os.path.join(self.low_res_folder, date_str + '.nc4')
            if os.path.exists(low_file):
                valid_info.append((date_str, low_file))
        
        print(f"유효한 저해상도 파일: {len(valid_info)} / 전체 기간: {len(self.dates)}일")
        return valid_info

    def _load_nc(self, path, var):
        try:
            with ncDataset(path) as nc:
                arr = nc.variables[var][:].astype(np.float32)
                if arr.ndim == 3:
                    arr = arr.squeeze(0)
                return arr
        except Exception as e:
            print(f"NC 로드 오류({path}): {e}")
            return np.zeros((61, 101), dtype=np.float32)

    def _load_and_resize(self, path, var, size):
        arr = self._load_nc(path, var)
        if arr.shape != size:
            t = torch.from_numpy(arr).float().unsqueeze(0).unsqueeze(0)
            resized = torch.nn.functional.interpolate(t, size=size, mode='bilinear', align_corners=False)
            return resized.squeeze().numpy()
        return arr

# ======================================================
# 모델 정의
# ======================================================
class Generator(nn.Module):
    def __init__(self, in_ch=2, n_layers=4, base_filters=64):
        super().__init__()
        layers = []
        c = in_ch
        for i in range(n_layers):
            out_filters = base_filters * (2**i)
            layers += [nn.Conv2d(c, out_filters, 3, 1, 1, bias=False), nn.BatchNorm2d(out_filters), nn.ReLU(True)]
            c = out_filters
        
        self.encoder = nn.Sequential(*layers)
        self.final = nn.Conv2d(c, 1, 3, 1, 1)

    def forward(self, x):
        x = self.encoder(x)
        x = self.final(x)
        return x

class Discriminator(nn.Module):
    def __init__(self, n_layers=4, base_filters=64):
        super().__init__()
        layers = [nn.Conv2d(1, base_filters, 4, 2, 1), nn.LeakyReLU(0.2, inplace=True)]
        
        c_in = base_filters
        for i in range(1, n_layers):
            c_out = base_filters * (2**i)
            layers += [nn.Conv2d(c_in, c_out, 4, 2, 1, bias=False), nn.BatchNorm2d(c_out), nn.LeakyReLU(0.2, inplace=True)]
            c_in = c_out

        layers.append(nn.Conv2d(c_in, 1, 1, 1, 0))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

# ======================================================
# NetCDF 저장 함수
# ======================================================
def save_to_netcdf(data, date_str, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    
    output_file = os.path.join(output_dir, f"{date_str}_Downscaled_Precipitation.nc")
    
    with ncDataset(output_file, 'w', format='NETCDF4') as nc:
        nc.createDimension('lat', 61)
        nc.createDimension('lon', 101)
        nc.createDimension('time', 1)
        
        time_var = nc.createVariable('time', 'f4', ('time',))
        time_var.units = f'days since {date_str[:4]}-{date_str[4:6]}-{date_str[6:8]} 00:00:00'
        time_var.calendar = 'gregorian'
        time_var[:] = [0]
        
        precip_var = nc.createVariable('precipitation', 'f4', ('time', 'lat', 'lon'), 
                                       fill_value=-9999.0, zlib=True, complevel=4)
        precip_var[:] = data.reshape(1, 61, 101)
        precip_var.units = 'mm/day'
        precip_var.long_name = 'Downscaled Precipitation'
        
        nc.title = f'Downscaled Precipitation for {date_str}'
        nc.institution = 'Generated by GAN Downscaling Model'
        nc.history = f'Created on {datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")}'

# ======================================================
# 최종 학습 및 평가 (수정된 버전)
# ======================================================
def train_final_model_fixed(train_ds, val_ds, test_ds, best_params, norm_stats, log_dir, epochs=50):
    G = Generator(2, best_params['n_layers_g'], best_params['n_filters_base_g']).to(DEVICE)
    D = Discriminator(best_params['n_layers_d'], best_params['n_filters_base_d']).to(DEVICE)
    
    opt_name = best_params['optimizer']
    OptimG = getattr(optim, opt_name)(G.parameters(), lr=best_params['lr_g'], betas=(0.5, 0.999))
    OptimD = getattr(optim, opt_name)(D.parameters(), lr=best_params['lr_d'], betas=(0.5, 0.999))
    
    criterion_gan = nn.BCEWithLogitsLoss()
    criterion_content = nn.L1Loss() if best_params['loss_function'] == 'L1' else nn.MSELoss()

    scaler_g = GradScaler('cuda', enabled=torch.cuda.is_available())
    scaler_d = GradScaler('cuda', enabled=torch.cuda.is_available())

    train_loader = DataLoader(train_ds, batch_size=best_params['batch_size'], shuffle=True, pin_memory=PIN_MEMORY, num_workers=NUM_WORKERS)
    val_loader = DataLoader(val_ds, batch_size=best_params['batch_size'], shuffle=False, pin_memory=PIN_MEMORY, num_workers=NUM_WORKERS)
    
    best_val_loss = float('inf')
    patience_counter = 0
    patience = 15
    min_delta = 1e-5

    for epoch in range(1, epochs + 1):
        G.train()
        D.train()
        for x, y in tqdm(train_loader, desc=f"Final Train E{epoch}", leave=False, dynamic_ncols=True):
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            
            # --- Discriminator 학습 (동적 라벨 크기) ---
            OptimD.zero_grad(set_to_none=True)
            with autocast('cuda', enabled=torch.cuda.is_available()):
                fake_img = G(x).detach()
                real_output = D(y)
                fake_output = D(fake_img)
                
                # 동적으로 라벨 크기 맞추기
                real_labels = torch.full(real_output.shape, 0.9, device=DEVICE)
                fake_labels = torch.full(fake_output.shape, 0.1, device=DEVICE)
                
                loss_real = criterion_gan(real_output, real_labels)
                loss_fake = criterion_gan(fake_output, fake_labels)
                d_loss = 0.5 * (loss_real + loss_fake)
            
            scaler_d.scale(d_loss).backward()
            scaler_d.step(OptimD)
            scaler_d.update()

            # --- Generator 학습 ---
            OptimG.zero_grad(set_to_none=True)
            with autocast('cuda', enabled=torch.cuda.is_available()):
                fake_img_for_g = G(x)
                output_for_g = D(fake_img_for_g)
                # Generator는 진짜라고 속이려 하므로 real_labels와 같은 크기
                real_labels_for_g = torch.full(output_for_g.shape, 0.9, device=DEVICE)
                g_adv_loss = criterion_gan(output_for_g, real_labels_for_g)
                g_content_loss = criterion_content(fake_img_for_g, y)
                g_loss = g_adv_loss + best_params['content_weight'] * g_content_loss
            
            scaler_g.scale(g_loss).backward()
            scaler_g.step(OptimG)
            scaler_g.update()

        # --- 검증 및 조기 종료 ---
        current_val_loss = 0.0
        G.eval()
        with torch.no_grad():
            for x_val, y_val in tqdm(val_loader, desc=f"Final Val E{epoch}", leave=False, dynamic_ncols=True):
                x_val, y_val = x_val.to(DEVICE), y_val.to(DEVICE)
                with autocast('cuda', enabled=torch.cuda.is_available()):
                    pred = G(x_val)
                    current_val_loss += criterion_content(pred, y_val).item()
        
        current_val_loss /= len(val_loader)
        logging.info(f"Epoch {epoch}/{epochs}, Validation Loss: {current_val_loss:.6f}")

        if current_val_loss < best_val_loss - min_delta:
            best_val_loss = current_val_loss
            patience_counter = 0
            torch.save(G.state_dict(), os.path.join(log_dir, 'best_generator.pth'))
            logging.info(f"Validation loss improved. Saved new best model.")
        else:
            patience_counter += 1
            logging.info(f"Validation loss did not improve. Patience: {patience_counter}/{patience}")
            if patience_counter >= patience:
                logging.info("Early stopping triggered.")
                break
    
    # --- 테스트 평가 ---
    G.load_state_dict(torch.load(os.path.join(log_dir, 'best_generator.pth'), weights_only=True))
    test_loader = DataLoader(test_ds, batch_size=1, shuffle=False)
    all_mse, all_mae = [], []
    
    high_res_mean = norm_stats['high_res']['mean']
    high_res_std = norm_stats['high_res']['std']

    G.eval()
    with torch.no_grad():
        for x_test, y_test in tqdm(test_loader, desc='Final Test Evaluation', dynamic_ncols=True):
            x_test, y_test = x_test.to(DEVICE), y_test.to(DEVICE)
            pred = G(x_test)
            
            y_unnormalized = (y_test.cpu().numpy() * high_res_std) + high_res_mean
            pred_unnormalized = (pred.cpu().numpy() * high_res_std) + high_res_mean
            
            all_mse.append(mean_squared_error(y_unnormalized.flatten(), pred_unnormalized.flatten()))
            all_mae.append(mean_absolute_error(y_unnormalized.flatten(), pred_unnormalized.flatten()))

    mean_mse = np.mean(all_mse)
    mean_mae = np.mean(all_mae)
    return G, D, mean_mse, mean_mae

# ======================================================
# 다운스케일링 함수
# ======================================================
def run_inference_and_save(model_path, normalization_stats, 
                          low_res_dir, dem_file, output_dir,
                          start_date, end_date, 
                          batch_size=32, device='cuda'):
    
    if torch.cuda.is_available() and device == 'cuda':
        device = torch.device('cuda')
        print(f"✅ GPU 사용: {torch.cuda.get_device_name(0)}")
    else:
        device = torch.device('cpu')
        print("❌ CPU 사용")
    
    print("학습된 모델 로딩 중...")
    model = Generator(in_ch=2, n_layers=4, base_filters=32).to(device)
    model.load_state_dict(torch.load(model_path, map_location=device, weights_only=True))
    model.eval()
    print("✅ 모델 로딩 완료")
    
    print("추론용 데이터셋 생성 중...")
    inference_dataset = InferenceRainfallDataset(
        low_res_dir, dem_file, start_date, end_date, normalization_stats
    )
    
    inference_loader = DataLoader(
        inference_dataset, 
        batch_size=batch_size, 
        shuffle=False, 
        num_workers=2, 
        pin_memory=True if device.type == 'cuda' else False
    )
    
    high_res_mean = normalization_stats['high_res']['mean']
    high_res_std = normalization_stats['high_res']['std']
    
    print(f"다운스케일링 시작... (총 {len(inference_dataset)}개 파일)")
    
    processed_count = 0
    failed_count = 0
    
    with torch.no_grad():
        for batch_idx, (batch_x, batch_dates) in enumerate(tqdm(inference_loader, desc="Downscaling Progress")):
            try:
                batch_x = batch_x.to(device, non_blocking=True)
                
                with autocast('cuda', enabled=(device.type == 'cuda')):
                    batch_pred = model(batch_x)
                
                batch_pred = batch_pred.cpu().numpy()
                batch_pred_unnorm = (batch_pred * high_res_std) + high_res_mean
                
                for i, date_str in enumerate(batch_dates):
                    try:
                        pred_data = batch_pred_unnorm[i, 0]
                        pred_data = np.clip(pred_data, 0, None)
                        
                        save_to_netcdf(pred_data, date_str, output_dir)
                        processed_count += 1
                        
                    except Exception as e:
                        print(f"❌ 저장 실패 ({date_str}): {e}")
                        failed_count += 1
                        
            except Exception as e:
                print(f"❌ 배치 {batch_idx} 추론 실패: {e}")
                failed_count += len(batch_dates)
    
    print(f"\n{'='*50}")
    print(f"🎉 다운스케일링 완료!")
    print(f"✅ 성공: {processed_count}개 파일")
    if failed_count > 0:
        print(f"❌ 실패: {failed_count}개 파일")
    print(f"📁 출력 디렉토리: {output_dir}")
    print(f"{'='*50}")

# ======================================================
# 메인 실행 함수
# ======================================================
def main():
    set_seed(42)
    run_date_str = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
    log_dir = setup_directories(run_date_str)
    setup_logging(log_dir)

    global DEVICE
    DEVICE = setup_gpu()
    
    # 경로 설정
    low_res_dir = '/mnt/c/Yeonji/GPM/Input'
    high_res_dir = '/mnt/c/Yeonji/2025.01.Drought/2004/1.CoKriging(Daily)'
    dem_file = '/mnt/c/Yeonji/2025.01.Drought/DEM_jeju.nc'
    
    for p in [low_res_dir, high_res_dir, dem_file]:
        if not os.path.exists(p):
            logging.error(f"경로가 존재하지 않습니다: {p}")
            raise FileNotFoundError(f"경로가 존재하지 않습니다: {p}")
    
    print("데이터셋 로딩 중...")
    train_ds = RainfallDataset(low_res_dir, high_res_dir, dem_file,
                               datetime.date(2004, 1, 1), datetime.date(2018, 12, 31))
    val_ds = RainfallDataset(low_res_dir, high_res_dir, dem_file,
                             datetime.date(2019, 1, 1), datetime.date(2021, 12, 31),
                             normalization_stats=train_ds.stats)
    test_ds = RainfallDataset(low_res_dir, high_res_dir, dem_file,
                              datetime.date(2022, 1, 1), datetime.date(2023, 12, 31),
                              normalization_stats=train_ds.stats)
    print("데이터셋 로딩 완료.")

    # 이전 최적화 결과 사용
    print("이전 하이퍼파라미터 최적화 결과를 사용합니다.")
    best_params = {
        'batch_size': 32, 
        'lr_g': 1.623702100615101e-05, 
        'lr_d': 5.740122018683213e-05, 
        'content_weight': 137.93587388434258, 
        'n_layers_g': 4, 
        'n_layers_d': 4, 
        'n_filters_base_g': 32, 
        'n_filters_base_d': 64, 
        'optimizer': 'AdamW', 
        'loss_function': 'L1'
    }
    best_value = 0.347622
    
    print(f"\n최적화 완료! Best Validation Loss: {best_value:.6f}")
    print(f"최적 하이퍼파라미터: {best_params}")
    logging.info(f"Using pre-computed best trial: {best_value} with params: {best_params}")

    print("\n최적 하이퍼파라미터로 최종 모델 학습 시작...")
    _, _, mse, mae = train_final_model_fixed(train_ds, val_ds, test_ds, best_params, train_ds.stats, log_dir, epochs=50)
    
    print("\n" + "="*50)
    print(f"=== 최종 학습 및 평가 완료 ===")
    print(f"Test MSE: {mse:.6f}")
    print(f"Test MAE: {mae:.6f}")
    print("="*50)
    logging.info(f"Final Test MSE: {mse:.6f}, MAE: {mae:.6f}")

    # ======================================================
    # 🎯 다운스케일링 실행
    # ======================================================
    print("\n" + "="*50)
    print("🚀 2004-2023년 전체 기간 다운스케일링 시작...")
    print("="*50)
    
    model_path = os.path.join(log_dir, 'best_generator.pth')
    output_dir = os.path.join(log_dir, 'downscaled_results_2004_2023')
    
    run_inference_and_save(
        model_path=model_path,
        normalization_stats=train_ds.stats,
        low_res_dir=low_res_dir,
        dem_file=dem_file,
        output_dir=output_dir,
        start_date=datetime.date(2004, 1, 1),
        end_date=datetime.date(2023, 12, 31),
        batch_size=32,
        device='cuda'
    )
    
    print("\n" + "="*60)
    print("🎉 모든 과정이 완료되었습니다!")
    print(f"📁 학습된 모델: {model_path}")
    print(f"📁 다운스케일링 결과: {output_dir}")
    print(f"📊 최종 성능 - MSE: {mse:.6f}, MAE: {mae:.6f}")
    print("="*60)

# ======================================================
# 통계 저장 및 로드 함수 (추가 기능)
# ======================================================
def save_normalization_stats(stats, file_path):
    """정규화 통계를 파일로 저장"""
    import json
    with open(file_path, 'w') as f:
        json.dump(stats, f, indent=2)
    print(f"✅ 정규화 통계 저장: {file_path}")

def load_normalization_stats(file_path):
    """저장된 정규화 통계 로드"""
    import json
    try:
        with open(file_path, 'r') as f:
            stats = json.load(f)
        print(f"✅ 정규화 통계 로드: {file_path}")
        return stats
    except FileNotFoundError:
        print(f"❌ 통계 파일을 찾을 수 없습니다: {file_path}")
        return None

# ======================================================
# 개별 다운스케일링 함수 (필요시 사용)
# ======================================================
def downscale_single_year(model_path, normalization_stats, 
                         low_res_dir, dem_file, output_dir, year):
    """특정 연도만 다운스케일링"""
    start_date = datetime.date(year, 1, 1)
    end_date = datetime.date(year, 12, 31)
    
    year_output_dir = os.path.join(output_dir, f"year_{year}")
    
    print(f"🗓️ {year}년 다운스케일링 시작...")
    
    run_inference_and_save(
        model_path=model_path,
        normalization_stats=normalization_stats,
        low_res_dir=low_res_dir,
        dem_file=dem_file,
        output_dir=year_output_dir,
        start_date=start_date,
        end_date=end_date,
        batch_size=64,  # 단일 연도이므로 더 큰 배치 사용 가능
        device='cuda'
    )

# ======================================================
# 결과 검증 함수
# ======================================================
def validate_downscaling_results(output_dir):
    """다운스케일링 결과 검증"""
    import glob
    
    nc_files = glob.glob(os.path.join(output_dir, "*.nc"))
    
    print(f"\n📊 다운스케일링 결과 검증")
    print(f"📁 출력 디렉토리: {output_dir}")
    print(f"📄 생성된 파일 수: {len(nc_files)}")
    
    if len(nc_files) > 0:
        print(f"📅 첫 번째 파일: {os.path.basename(nc_files[0])}")
        print(f"📅 마지막 파일: {os.path.basename(nc_files[-1])}")
        
        # 파일 크기 확인
        total_size = sum(os.path.getsize(f) for f in nc_files)
        print(f"💾 총 파일 크기: {total_size / (1024**3):.2f} GB")
        
        # 샘플 파일 내용 확인
        try:
            sample_file = nc_files[len(nc_files)//2]
            with ncDataset(sample_file, 'r') as nc:
                precip_data = nc.variables['precipitation'][:]
                print(f"📐 데이터 형태: {precip_data.shape}")
                print(f"📊 강수량 범위: {precip_data.min():.3f} ~ {precip_data.max():.3f} mm/day")
                print(f"📊 평균 강수량: {precip_data.mean():.3f} mm/day")
        except Exception as e:
            print(f"❌ 샘플 파일 읽기 실패: {e}")
    else:
        print("❌ 생성된 파일이 없습니다!")
    
    return len(nc_files)

# ======================================================
# 실행 예시 함수들
# ======================================================
def run_training_only():
    """학습만 실행 (다운스케일링 제외)"""
    print("🎯 학습만 실행합니다...")
    # main() 함수에서 다운스케일링 부분만 제외하여 실행
    pass

def run_downscaling_only(model_path, stats_path=None):
    """다운스케일링만 실행 (기존 모델 사용)"""
    print("🎯 다운스케일링만 실행합니다...")
    
    # 설정
    low_res_dir = '/mnt/c/Yeonji/GPM/Input'
    dem_file = '/mnt/c/Yeonji/2025.01.Drought/DEM_jeju.nc'
    output_dir = './downscaling_only_results'
    
    # 정규화 통계 로드
    if stats_path and os.path.exists(stats_path):
        normalization_stats = load_normalization_stats(stats_path)
    else:
        print("⚠️ 정규화 통계 파일이 없습니다. 기본값을 사용합니다.")
        normalization_stats = {
            'low_res': {'mean': 0.7516844868659973, 'std': 3.591147661209106},
            'high_res': {'mean': 0.7516844868659973, 'std': 3.591147661209106}
        }
    
    # 다운스케일링 실행
    run_inference_and_save(
        model_path=model_path,
        normalization_stats=normalization_stats,
        low_res_dir=low_res_dir,
        dem_file=dem_file,
        output_dir=output_dir,
        start_date=datetime.date(2004, 1, 1),
        end_date=datetime.date(2023, 12, 31),
        batch_size=32,
        device='cuda'
    )
    
    # 결과 검증
    validate_downscaling_results(output_dir)

if __name__ == '__main__':
    main()

## 0713(DEM 바꿈)

In [ ]:
import xarray as xr

ds = xr.open_dataset('/mnt/c/Yeonji/2025.01.Drought/DEM_jeju.nc')

# 전체 요약
print(ds)

# 전역 속성
print("Global Attributes:")
print(ds.attrs)

# 변수별 속성
for var in ds.data_vars:
    print(f"\nVariable: {var}")
    print(ds[var].attrs)

In [ ]:
import os
import random
import datetime
import logging
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
from netCDF4 import Dataset as ncDataset
from tqdm import tqdm
from sklearn.metrics import mean_squared_error, mean_absolute_error
import xarray as xr

# ─────────────────────────────────────────────────────────
# 0) 전역 설정
# ─────────────────────────────────────────────────────────
NUM_WORKERS = 2
PIN_MEMORY   = True

# 제주 영역: 0.01° 해상도 (목표 해상도)
JEJU_LAT_START, JEJU_LAT_END = 33.1, 33.7
JEJU_LON_START, JEJU_LON_END = 126.1, 127.1
RESOLUTION = 0.01
LAT_VALS  = np.arange(JEJU_LAT_START, JEJU_LAT_END + RESOLUTION/2, RESOLUTION, dtype='f4')
LON_VALS  = np.arange(JEJU_LON_START, JEJU_LON_END + RESOLUTION/2, RESOLUTION, dtype='f4')

print(f"Target grid size: {LAT_VALS.size} x {LON_VALS.size}")

# ─────────────────────────────────────────────────────────
# 1) DEM 데이터 로딩 및 전처리 함수 (수정됨)
# ─────────────────────────────────────────────────────────
def load_dem_data(dem_file):
    """DEM 데이터를 목표 그리드에 맞게 로딩 및 리샘플링"""
    try:
        # xarray로 DEM 파일 로드
        dem_ds = xr.open_dataset(dem_file)
        print(f"DEM 원본 차원: {dem_ds.dims}")
        
        # elevation 데이터 추출
        if 'elevation' in dem_ds.data_vars:
            elevation = dem_ds['elevation'].values
            print(f"Elevation shape: {elevation.shape}")
            
            # 좌표 정보 확인
            if 'latitude' in dem_ds.coords and 'longitude' in dem_ds.coords:
                dem_lat = dem_ds['latitude'].values
                dem_lon = dem_ds['longitude'].values
                print(f"DEM 실제 범위 - Lat: {dem_lat.min():.3f}~{dem_lat.max():.3f}, "
                      f"Lon: {dem_lon.min():.3f}~{dem_lon.max():.3f}")
            
            # 목표 그리드 크기로 리샘플링
            target_shape = (LAT_VALS.size, LON_VALS.size)
            elevation_tensor = torch.tensor(elevation, dtype=torch.float32).unsqueeze(0).unsqueeze(0)
            elevation_resampled = torch.nn.functional.interpolate(
                elevation_tensor, size=target_shape, mode='bilinear', align_corners=False
            ).squeeze().numpy()
            
            print(f"DEM 리샘플링 완료: {elevation.shape} -> {elevation_resampled.shape}")
            return elevation_resampled.astype(np.float32)
        else:
            raise ValueError("DEM 파일에서 'elevation' 변수를 찾을 수 없습니다.")
            
    except Exception as e:
        print(f"DEM 로딩 중 오류: {e}")
        # 대안: NetCDF4로 직접 로드 시도
        try:
            with ncDataset(dem_file, 'r') as nc:
                elevation = nc.variables['elevation'][:].astype(np.float32)
                elevation = np.squeeze(elevation)
                
                # 리샘플링
                target_shape = (LAT_VALS.size, LON_VALS.size)
                elevation_tensor = torch.tensor(elevation, dtype=torch.float32).unsqueeze(0).unsqueeze(0)
                elevation_resampled = torch.nn.functional.interpolate(
                    elevation_tensor, size=target_shape, mode='bilinear', align_corners=False
                ).squeeze().numpy()
                
                print(f"NetCDF4로 DEM 로딩 완료: {elevation.shape} -> {elevation_resampled.shape}")
                return elevation_resampled.astype(np.float32)
        except Exception as e2:
            print(f"NetCDF4로도 DEM 로딩 실패: {e2}")
            raise

# ─────────────────────────────────────────────────────────
# 2) NetCDF 저장 함수 (위·경도 좌표 포함)
# ─────────────────────────────────────────────────────────
def save_to_netcdf(data, date_str, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    outfn = os.path.join(output_dir, f"{date_str}_Downscaled_Precipitation.nc")
    n_lat, n_lon = LAT_VALS.size, LON_VALS.size
    time_unit    = f"days since {date_str[:4]}-{date_str[4:6]}-{date_str[6:]} 00:00:00"
    
    with ncDataset(outfn, 'w', format='NETCDF4') as ds:
        # 차원
        ds.createDimension('time', 1)
        ds.createDimension('lat',  n_lat)
        ds.createDimension('lon',  n_lon)
        # time 변수
        tv = ds.createVariable('time', 'f4', ('time',))
        tv[:]        = [0.0]
        tv.units     = time_unit
        tv.calendar  = 'gregorian'
        # lat 변수
        lv = ds.createVariable('lat', 'f4', ('lat',))
        lv[:]        = LAT_VALS
        lv.units     = 'degrees_north'
        lv.standard_name = 'latitude'
        lv.axis      = 'Y'
        # lon 변수
        lov = ds.createVariable('lon', 'f4', ('lon',))
        lov[:]       = LON_VALS
        lov.units    = 'degrees_east'
        lov.standard_name = 'longitude'
        lov.axis     = 'X'
        # precipitation 변수
        p = ds.createVariable('precipitation','f4',
                              ('time','lat','lon'),
                              zlib=True, complevel=4,
                              fill_value=-9999.0)
        p[0,:,:]       = data.astype('f4')
        p.units        = 'mm/day'
        p.long_name    = 'Downscaled Precipitation'
        p.coordinates  = 'time lat lon'
        p.grid_mapping = 'crs'
        # CRS 변수
        crs = ds.createVariable('crs','i4')
        crs.grid_mapping_name  = 'latitude_longitude'
        crs.epsg_code          = 'EPSG:4326'
        crs.semi_major_axis    = 6378137.0
        crs.inverse_flattening = 298.257223563
        # 전역 속성
        ds.Conventions = 'CF-1.8'
        ds.title       = f"Downscaled Precipitation for {date_str}"
        ds.history     = f"Created on {datetime.datetime.now():%Y-%m-%d %H:%M:%S}"

# ─────────────────────────────────────────────────────────
# 3) Dataset 클래스 (DEM 로딩 개선)
# ─────────────────────────────────────────────────────────
class RainfallDataset(Dataset):
    def __init__(self, low_res_folder, high_res_folder, dem_file,
                 start_date, end_date, normalization_stats=None):
        self.low = low_res_folder
        self.high = high_res_folder
        self.dates = [start_date + datetime.timedelta(days=i)
                      for i in range((end_date - start_date).days + 1)]
        # 유효한 파일 쌍 수집
        self.paths = []
        for d in self.dates:
            f1 = os.path.join(self.low,  d.strftime('%Y%m%d') + '.nc4')
            f2 = os.path.join(self.high, d.strftime('%Y%m%d') + '_Cokriging_Precipitation.nc')
            if os.path.exists(f1) and os.path.exists(f2):
                self.paths.append((f1,f2))
        if not self.paths:
            raise ValueError("유효한 데이터가 없습니다.")
        
        print(f"유효한 데이터 쌍: {len(self.paths)}개")
        
        # DEM 로딩 (개선된 함수 사용)
        self.dem = load_dem_data(dem_file)
        print(f"DEM 최종 크기: {self.dem.shape}")
        
        # 정규화 통계 계산 또는 사용
        self.stats = normalization_stats or self._calc_stats()
        
    def __len__(self):
        return len(self.paths)
        
    def __getitem__(self, idx):
        low_f, high_f = self.paths[idx]
        low  = self._load_and_resize(low_f, 'precipitation',
                                     (LAT_VALS.size, LON_VALS.size))
        high = self._load_nc(high_f, 'precipitation')
        
        # 정규화
        low  = (low  - self.stats['low_res']['mean']) / (self.stats['low_res']['std'] + 1e-8)
        high = (high - self.stats['high_res']['mean'])/(self.stats['high_res']['std']+1e-8)
        dem_norm = (self.dem - self.stats['dem']['mean']) / (self.stats['dem']['std'] + 1e-8)
        
        x = torch.tensor(np.stack([low, dem_norm], axis=0), dtype=torch.float32)
        y = torch.tensor(high, dtype=torch.float32).unsqueeze(0)
        return x, y
        
    def _load_nc(self, path, var):
        with ncDataset(path) as nc:
            arr = nc.variables[var][:].astype(np.float32)
            return arr.squeeze() if arr.ndim==3 else arr
            
    def _load_and_resize(self, path, var, size):
        arr = self._load_nc(path, var)
        if arr.shape != size:
            t = torch.tensor(arr, dtype=torch.float32).unsqueeze(0).unsqueeze(0)
            arr = torch.nn.functional.interpolate(
                t, size=size, mode='bilinear', align_corners=False
            ).squeeze().numpy()
        return arr
        
    def _calc_stats(self):
        print("정규화 통계 계산 중...")
        lows, highs = [], []
        for i, (l, h) in enumerate(self.paths):
            if i % 100 == 0:
                print(f"처리 중: {i}/{len(self.paths)}")
            lows.append(self._load_and_resize(l, 'precipitation',
                         (LAT_VALS.size, LON_VALS.size)))
            highs.append(self._load_nc(h, 'precipitation'))
            
        lows  = np.stack(lows)
        highs = np.stack(highs)
        
        stats = {
            'low_res': {'mean': lows.mean(),  'std': lows.std()},
            'high_res': {'mean': highs.mean(), 'std': highs.std()},
            'dem': {'mean': self.dem.mean(), 'std': self.dem.std()}
        }
        
        print("정규화 통계:")
        for key, val in stats.items():
            print(f"  {key}: mean={val['mean']:.4f}, std={val['std']:.4f}")
            
        return stats

# ─────────────────────────────────────────────────────────
# 4) Generator 모델 정의 (단순화)
# ─────────────────────────────────────────────────────────
class Generator(nn.Module):
    def __init__(self, in_ch=2, n_layers=4, base_filters=32):
        super().__init__()
        layers = []
        c = in_ch
        for i in range(n_layers):
            f = base_filters * (2**i)
            layers += [nn.Conv2d(c,f,3,1,1,bias=False),
                       nn.BatchNorm2d(f),
                       nn.ReLU(True)]
            c = f
        self.encoder = nn.Sequential(*layers)
        self.final   = nn.Conv2d(c,1,3,1,1)
        
    def forward(self,x):
        return self.final(self.encoder(x))

# ─────────────────────────────────────────────────────────
# 5) 최종 학습 및 평가 (하이퍼옵션 생략)
# ─────────────────────────────────────────────────────────
def train_final(train_ds, val_ds, test_ds, params, log_dir, epochs=50):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")
    
    G = Generator(2, params['n_layers_g'], params['n_filters_base_g']).to(device)
    optimG = optim.AdamW(G.parameters(), lr=params['lr_g'], betas=(0.5,0.999))
    criterion = nn.L1Loss() if params['loss_function']=='L1' else nn.MSELoss()
    scaler = GradScaler(enabled=torch.cuda.is_available())
    
    train_loader = DataLoader(train_ds, batch_size=params['batch_size'], shuffle=True,
                              pin_memory=PIN_MEMORY, num_workers=NUM_WORKERS)
    val_loader   = DataLoader(val_ds,   batch_size=params['batch_size'], shuffle=False,
                              pin_memory=PIN_MEMORY, num_workers=NUM_WORKERS)
    
    best_loss, counter = 1e9, 0
    for ep in range(1, epochs+1):
        G.train()
        train_loss = 0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimG.zero_grad()
            with autocast(device_type='cuda' if torch.cuda.is_available() else 'cpu', enabled=True):
                out = G(x)
                loss = criterion(out, y) * params['content_weight']
            scaler.scale(loss).backward()
            scaler.step(optimG)
            scaler.update()
            train_loss += loss.item()
            
        train_loss /= len(train_loader)
        
        # 검증
        G.eval()
        vloss = 0
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(device), y.to(device)
                vloss += criterion(G(x), y).item()
        vloss /= len(val_loader)
        
        logging.info(f"Epoch {ep}, Train Loss: {train_loss:.6f}, Val Loss: {vloss:.6f}")
        print(f"Epoch {ep}, Train Loss: {train_loss:.6f}, Val Loss: {vloss:.6f}")
        
        if vloss < best_loss:
            best_loss, counter = vloss, 0
            torch.save(G.state_dict(), os.path.join(log_dir,'best_generator.pth'))
        else:
            counter += 1
            if counter >= 15:
                print(f"Early stopping at epoch {ep}")
                break
                
    # 테스트
    G.load_state_dict(torch.load(os.path.join(log_dir,'best_generator.pth')))
    test_loader = DataLoader(test_ds, batch_size=1, shuffle=False)
    mse_list, mae_list = [], []
    
    G.eval()
    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(device), y.to(device)
            pred = G(x)
            y_, p_ = y.cpu().numpy().flatten(), pred.cpu().numpy().flatten()
            mse_list.append(mean_squared_error(y_, p_))
            mae_list.append(mean_absolute_error(y_, p_))
            
    return G, np.mean(mse_list), np.mean(mae_list)

# ─────────────────────────────────────────────────────────
# 6) 추론 및 저장 (수정됨)
# ─────────────────────────────────────────────────────────
def run_inference(model, train_dataset, low_res_dir, output_dir, start_date, end_date):
    """추론 실행 및 결과 저장"""
    device = next(model.parameters()).device
    model.eval()
    
    # DEM 데이터 (정규화된 버전 사용)
    dem_norm = (train_dataset.dem - train_dataset.stats['dem']['mean']) / \
               (train_dataset.stats['dem']['std'] + 1e-8)
    
    dates = [start_date + datetime.timedelta(days=i)
             for i in range((end_date - start_date).days+1)]
             
    print(f"추론 시작: {len(dates)}일 처리")
    
    for i, date in enumerate(tqdm(dates, desc="Inference")):
        lf = os.path.join(low_res_dir, date.strftime('%Y%m%d') + '.nc4')
        if not os.path.exists(lf):
            continue
            
        # 저해상도 데이터 로딩
        with ncDataset(lf) as nc:
            low_arr = nc.variables['precipitation'][:].astype(np.float32)
            low_arr = low_arr.squeeze() if low_arr.ndim == 3 else low_arr
            
        # 목표 해상도로 리샘플링
        if low_arr.shape != (LAT_VALS.size, LON_VALS.size):
            low_tensor = torch.tensor(low_arr, dtype=torch.float32).unsqueeze(0).unsqueeze(0)
            low_arr = torch.nn.functional.interpolate(
                low_tensor, size=(LAT_VALS.size, LON_VALS.size), 
                mode='bilinear', align_corners=False
            ).squeeze().numpy()
            
        # 정규화
        low_norm = (low_arr - train_dataset.stats['low_res']['mean']) / \
                   (train_dataset.stats['low_res']['std'] + 1e-8)
        
        # 입력 텐서 생성
        x = torch.tensor(np.stack([low_norm, dem_norm], axis=0), 
                        dtype=torch.float32).unsqueeze(0).to(device)
        
        # 예측
        with torch.no_grad():
            pred = model(x).cpu().squeeze().numpy()
            
        # 역정규화
        pred_denorm = pred * train_dataset.stats['high_res']['std'] + \
                     train_dataset.stats['high_res']['mean']
        
        # 음수 값 제거 (강수량은 0 이상)
        pred_denorm = np.maximum(pred_denorm, 0)
        
        # NetCDF로 저장
        save_to_netcdf(pred_denorm, date.strftime('%Y%m%d'), output_dir)

# ─────────────────────────────────────────────────────────
# 7) 메인 실행
# ─────────────────────────────────────────────────────────
def main():
    random.seed(42)
    np.random.seed(42)
    torch.manual_seed(42)
    
    # 로그 디렉토리
    run_dt = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
    log_dir = f"/mnt/c/Yeonji/2025.01.Drought/2004/{run_dt}"
    os.makedirs(log_dir, exist_ok=True)
    
    logging.basicConfig(level=logging.INFO,
                        filename=os.path.join(log_dir,'training.log'),
                        format='%(asctime)s - %(message)s')
    
    # GPU 확인
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    logging.info(f"Using device: {device}")
    print(f"Using device: {device}")
    
    # 경로 설정 (절대경로)
    low_res_dir  = r"/mnt/c/Yeonji/GPM/Input"
    high_res_dir = r"/mnt/c/Yeonji/2025.01.Drought/2004/1.CoKriging(Daily)"
    dem_file     = r"/mnt/c/Yeonji/2025.01.Drought/DEM_jeju.nc"
    
    print("데이터셋 로딩 중...")
    
    # 데이터셋
    train_ds = RainfallDataset(low_res_dir, high_res_dir, dem_file,
                               datetime.date(2004,1,1), datetime.date(2018,12,31))
    val_ds   = RainfallDataset(low_res_dir, high_res_dir, dem_file,
                               datetime.date(2019,1,1), datetime.date(2021,12,31),
                               normalization_stats=train_ds.stats)
    test_ds  = RainfallDataset(low_res_dir, high_res_dir, dem_file,
                               datetime.date(2022,1,1), datetime.date(2023,12,31),
                               normalization_stats=train_ds.stats)
    
    print(f"데이터셋 크기 - Train: {len(train_ds)}, Val: {len(val_ds)}, Test: {len(test_ds)}")
    
    # 사전 정의된 최적 하이퍼파라미터
    best_params = {
        'batch_size':       32,
        'lr_g':             1.623702100615101e-05,
        'content_weight':   137.93587388434258,
        'n_layers_g':       4,
        'n_filters_base_g': 32,
        'loss_function':    'L1'
    }
    
    print("모델 학습 시작...")
    
    # 모델 학습
    model, mse, mae = train_final(train_ds, val_ds, test_ds,
                                  best_params, log_dir, epochs=50)
    print(f"▶ Test MSE: {mse:.4f}, MAE: {mae:.4f}")
    
    # 추론
    print("추론 시작...")
    out_folder = r"/mnt/c/Yeonji/2025.01.Drought/2004/0713"
    run_inference(model, train_ds, low_res_dir, out_folder,
                  datetime.date(2004,1,1), datetime.date(2023,12,31))
    print("✅ 전체 파이프라인 완료!")

if __name__=='__main__':
    main()

In [ ]:
import os
import random
import datetime
import logging
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
from netCDF4 import Dataset as ncDataset
from tqdm import tqdm
from sklearn.metrics import mean_squared_error, mean_absolute_error
import rasterio
from rasterio.enums import Resampling

# ─────────────────────────────────────────────────────────
# 0) 전역 설정
# ─────────────────────────────────────────────────────────
NUM_WORKERS = 2
PIN_MEMORY = True

# 제주 영역: 0.01° 해상도 (목표 격자)
JEJU_LAT_START, JEJU_LAT_END = 33.1, 33.7  # 61 셀
JEJU_LON_START, JEJU_LON_END = 126.1, 127.1
RESOLUTION = 0.01
LAT_VALS = np.arange(JEJU_LAT_START, JEJU_LAT_END + RESOLUTION/2, RESOLUTION, dtype='f4')
LON_VALS = np.arange(JEJU_LON_START, JEJU_LON_END + RESOLUTION/2, RESOLUTION, dtype='f4')

# ─────────────────────────────────────────────────────────
# 1) GeoTIFF DEM 로딩 함수
# ─────────────────────────────────────────────────────────
def load_dem_geotiff(dem_path, target_shape):
    """
    GeoTIFF DEM 파일을 bilinear 리샘플링하여 target_shape (height, width) 로 반환
    """
    with rasterio.open(dem_path) as src:
        data = src.read(
            1,
            out_shape=target_shape,
            resampling=Resampling.bilinear
        )
    return data.astype(np.float32)

# ─────────────────────────────────────────────────────────
# 2) NetCDF 저장 함수 (위·경도 좌표 포함)
# ─────────────────────────────────────────────────────────
def save_to_netcdf(data, date_str, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    outfn = os.path.join(output_dir, f"{date_str}_Downscaled_Precipitation.nc")
    n_lat, n_lon = LAT_VALS.size, LON_VALS.size
    time_unit = f"days since {date_str[:4]}-{date_str[4:6]}-{date_str[6:]} 00:00:00"
    with ncDataset(outfn, 'w', format='NETCDF4') as ds:
        ds.createDimension('time', 1)
        ds.createDimension('lat', n_lat)
        ds.createDimension('lon', n_lon)
        tv = ds.createVariable('time', 'f4', ('time',))
        tv[:] = [0.0]; tv.units = time_unit; tv.calendar = 'gregorian'
        lv = ds.createVariable('lat', 'f4', ('lat',))
        lv[:] = LAT_VALS; lv.units = 'degrees_north'; lv.standard_name = 'latitude'; lv.axis = 'Y'
        lov = ds.createVariable('lon', 'f4', ('lon',))
        lov[:] = LON_VALS; lov.units = 'degrees_east'; lov.standard_name = 'longitude'; lov.axis = 'X'
        p = ds.createVariable('precipitation','f4',('time','lat','lon'),
                              zlib=True, complevel=4, fill_value=-9999.0)
        p[0,:,:] = data.astype('f4')
        p.units = 'mm/day'; p.long_name = 'Downscaled Precipitation'
        p.coordinates = 'time lat lon'; p.grid_mapping = 'crs'
        crs = ds.createVariable('crs','i4')
        crs.grid_mapping_name = 'latitude_longitude'; crs.epsg_code = 'EPSG:4326'
        crs.semi_major_axis = 6378137.0; crs.inverse_flattening = 298.257223563
        ds.Conventions = 'CF-1.8'
        ds.title = f"Downscaled Precipitation for {date_str}"
        ds.history = f"Created on {datetime.datetime.now():%Y-%m-%d %H:%M:%S}"  

# ─────────────────────────────────────────────────────────
# 3) Dataset 클래스 (GeoTIFF DEM 사용)
# ─────────────────────────────────────────────────────────
class RainfallDataset(Dataset):
    def __init__(self, low_res_folder, high_res_folder, dem_file,
                 start_date, end_date, normalization_stats=None):
        self.low = low_res_folder
        self.high = high_res_folder
        self.dates = [start_date + datetime.timedelta(days=i)
                      for i in range((end_date - start_date).days + 1)]
        self.paths = []
        for d in self.dates:
            f1 = os.path.join(self.low, d.strftime('%Y%m%d') + '.nc4')
            f2 = os.path.join(self.high, d.strftime('%Y%m%d') + '_Cokriging_Precipitation.nc')
            if os.path.exists(f1) and os.path.exists(f2):
                self.paths.append((f1,f2))
        if not self.paths:
            raise ValueError("유효한 데이터가 없습니다.")
        print(f"유효한 데이터 쌍: {len(self.paths)}개")
        print("DEM GeoTIFF 로딩 및 재샘플링 중...")
        self.dem = load_dem_geotiff(
            dem_file,
            target_shape=(LAT_VALS.size, LON_VALS.size)
        )
        print(f"DEM 최종 크기: {self.dem.shape}")
        self.stats = normalization_stats or self._calc_stats()
    def __len__(self): return len(self.paths)
    def __getitem__(self, idx):
        low_f, high_f = self.paths[idx]
        low_arr = self._load_nc(low_f, 'precipitation')
        # 리샘플 if needed
        if low_arr.shape != (LAT_VALS.size, LON_VALS.size):
            t = torch.tensor(low_arr, dtype=torch.float32).unsqueeze(0).unsqueeze(0)
            low_arr = torch.nn.functional.interpolate(
                t, size=(LAT_VALS.size, LON_VALS.size),
                mode='bilinear', align_corners=False
            ).squeeze().numpy()
        high_arr = self._load_nc(high_f, 'precipitation')
        # 정규화
        low_norm = (low_arr - self.stats['low_res']['mean']) / (self.stats['low_res']['std'] + 1e-8)
        high_norm= (high_arr- self.stats['high_res']['mean'])/ (self.stats['high_res']['std']+1e-8)
        dem_norm = (self.dem - self.stats['dem']['mean']) / (self.stats['dem']['std']+1e-8)
        x = torch.tensor(np.stack([low_norm, dem_norm], axis=0), dtype=torch.float32)
        y = torch.tensor(high_norm, dtype=torch.float32).unsqueeze(0)
        return x, y
    def _load_nc(self, path, var):
        with ncDataset(path) as nc:
            arr = nc.variables[var][:].astype(np.float32)
            return arr.squeeze() if arr.ndim==3 else arr
    def _calc_stats(self):
        print("정규화 통계 계산 중...")
        lows, highs = [], []
        for i, (l, h) in enumerate(self.paths):
            if i % 100 == 0: print(f"통계[{i}/{len(self.paths)}]")
            lows.append(load_dem_geotiff(l, (LAT_VALS.size, LON_VALS.size)) if False else self._load_nc(l,'precipitation'))
            highs.append(self._load_nc(h,'precipitation'))
        lows  = np.stack([
            torch.nn.functional.interpolate(
                torch.tensor(a, dtype=torch.float32).unsqueeze(0).unsqueeze(0),
                size=(LAT_VALS.size,LON_VALS.size), mode='bilinear', align_corners=False
            ).squeeze().numpy() if a.shape!=(LAT_VALS.size,LON_VALS.size) else a
            for a in lows
        ])
        highs = np.stack(highs)
        stats = {
            'low_res':  {'mean': lows.mean(),  'std': lows.std()},
            'high_res': {'mean': highs.mean(), 'std': highs.std()},
            'dem':      {'mean': self.dem.mean(), 'std': self.dem.std()}
        }
        print("통계 완료:")
        for k,v in stats.items(): print(f" {k}: mean={v['mean']:.4f}, std={v['std']:.4f}")
        return stats

# ─────────────────────────────────────────────────────────
# 4) Generator 모델 정의
# ─────────────────────────────────────────────────────────
class Generator(nn.Module):
    def __init__(self, in_ch=2, n_layers=4, base_filters=32):
        super().__init__()
        layers=[]; c=in_ch
        for i in range(n_layers):
            f=base_filters*(2**i)
            layers += [nn.Conv2d(c,f,3,1,1,bias=False), nn.BatchNorm2d(f), nn.ReLU(True)]
            c=f
        self.encoder=nn.Sequential(*layers)
        self.final  =nn.Conv2d(c,1,3,1,1)
    def forward(self,x): return self.final(self.encoder(x))

# ─────────────────────────────────────────────────────────
# 5) 학습/테스트 함수 (하이퍼옵션 생략)
# ─────────────────────────────────────────────────────────
def train_final(train_ds, val_ds, test_ds, params, log_dir, epochs=50):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    logging.info(f"Using device: {device}")
    G=Generator(2, params['n_layers_g'], params['n_filters_base_g']).to(device)
    optimG=optim.AdamW(G.parameters(), lr=params['lr_g'], betas=(0.5,0.999))
    criterion=nn.L1Loss() if params['loss_function']=='L1' else nn.MSELoss()
    scaler=GradScaler(enabled=torch.cuda.is_available())
    tl=DataLoader(train_ds, batch_size=params['batch_size'], shuffle=True, pin_memory=PIN_MEMORY, num_workers=NUM_WORKERS)
    vl=DataLoader(val_ds,   batch_size=params['batch_size'], shuffle=False,pin_memory=PIN_MEMORY, num_workers=NUM_WORKERS)
    best, cnt=1e9,0
    for ep in range(1,epochs+1):
        G.train(); train_loss=0
        for x,y in tl:
            x,y=x.to(device),y.to(device)
            optimG.zero_grad()
            with autocast(device_type=device.type, enabled=torch.cuda.is_available()):
                loss=criterion(G(x),y)*params['content_weight']
            scaler.scale(loss).backward(); scaler.step(optimG); scaler.update()
            train_loss+=loss.item()
        train_loss/=len(tl)
        G.eval(); val_loss=0
        with torch.no_grad():
            for x,y in vl: val_loss+=criterion(G(x.to(device)),y.to(device)).item()
        val_loss/=len(vl)
        logging.info(f"Epoch {ep}, Train {train_loss:.6f}, Val {val_loss:.6f}")
        if val_loss<best: best,val_cnt=val_loss,0; torch.save(G.state_dict(), os.path.join(log_dir,'best_generator.pth'))
        else: cnt+=1;
        if cnt>=15: break
    G.load_state_dict(torch.load(os.path.join(log_dir,'best_generator.pth')))
    # 테스트
    tle=DataLoader(test_ds, batch_size=1, shuffle=False)
    ms,ma=[],[]
    with torch.no_grad():
        for x,y in tle:
            x,y=x.to(device),y.to(device)
            p=G(x)
            y_,p_=y.cpu().numpy().flatten(),p.cpu().numpy().flatten()
            ms.append(mean_squared_error(y_,p_)); ma.append(mean_absolute_error(y_,p_))
    return G, np.mean(ms), np.mean(ma)

# ─────────────────────────────────────────────────────────
# 6) 추론 및 저장
# ─────────────────────────────────────────────────────────
def run_inference(model, train_ds, low_res_dir, output_dir, start_date, end_date):
    device=next(model.parameters()).device
    model.eval()
    dem_norm=(train_ds.dem - train_ds.stats['dem']['mean'])/(train_ds.stats['dem']['std']+1e-8)
    dates=[start_date+datetime.timedelta(days=i) for i in range((end_date-start_date).days+1)]
    for date in tqdm(dates, desc="Inference"):
        lf=os.path.join(low_res_dir,date.strftime('%Y%m%d')+'.nc4')
        if not os.path.exists(lf): continue
        with ncDataset(lf) as nc: arr=nc.variables['precipitation'][:].astype(np.float32)
        arr=arr.squeeze() if arr.ndim==3 else arr
        if arr.shape!=(LAT_VALS.size,LON_VALS.size):
            t=torch.tensor(arr,dtype=torch.float32).unsqueeze(0).unsqueeze(0)
            arr=torch.nn.functional.interpolate(t,size=(LAT_VALS.size,LON_VALS.size),mode='bilinear',align_corners=False).squeeze().numpy()
        low_norm=(arr-train_ds.stats['low_res']['mean'])/(train_ds.stats['low_res']['std']+1e-8)
        x=torch.tensor(np.stack([low_norm,dem_norm],axis=0),dtype=torch.float32).unsqueeze(0).to(device)
        with torch.no_grad(): pred=model(x).cpu().squeeze().numpy()
        pred_den=pred*train_ds.stats['high_res']['std']+train_ds.stats['high_res']['mean']
        pred_den=np.clip(pred_den,0,None)
        save_to_netcdf(pred_den,date.strftime('%Y%m%d'),output_dir)

# ─────────────────────────────────────────────────────────
# 7) 메인 실행
# ─────────────────────────────────────────────────────────
def main():
    random.seed(42); np.random.seed(42); torch.manual_seed(42)
    run_dt=datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
    log_dir=f"/mnt/c/Yeonji/2025.01.Drought/2004/{run_dt}"
    os.makedirs(log_dir, exist_ok=True)
    logging.basicConfig(level=logging.INFO,filename=os.path.join(log_dir,'train.log'),format='%(asctime)s - %(message)s')
    low_res_dir=r"/mnt/c/Yeonji/GPM/Input"
    high_res_dir=r"/mnt/c/Yeonji/2025.01.Drought/2004/1.CoKriging(Daily)"
    dem_file=r"/mnt/c/Yeonji/2025.01.Drought/DEM(merge).tif"  # GeoTIFF
    print("데이터셋 로딩 중...")
    train_ds=RainfallDataset(low_res_dir,high_res_dir,dem_file,datetime.date(2004,1,1),datetime.date(2018,12,31))
    val_ds  =RainfallDataset(low_res_dir,high_res_dir,dem_file,datetime.date(2019,1,1),datetime.date(2021,12,31),normalization_stats=train_ds.stats)
    test_ds =RainfallDataset(low_res_dir,high_res_dir,dem_file,datetime.date(2022,1,1),datetime.date(2023,12,31),normalization_stats=train_ds.stats)
    print(f"Train/Val/Test sizes: {len(train_ds)}/{len(val_ds)}/{len(test_ds)}")
    best_params={'batch_size':32,'lr_g':1.6237e-05,'content_weight':137.9359,'n_layers_g':4,'n_filters_base_g':32,'loss_function':'L1'}
    print("모델 학습 시작...")
    model,mse,mae=train_final(train_ds,val_ds,test_ds,best_params,log_dir,epochs=50)
    print(f"▶ Test MSE: {mse:.4f}, MAE: {mae:.4f}")
    print("추론 및 저장 시작...")
    out_folder=r"/mnt/c/Yeonji/2025.01.Drought/2004/0713(2)"
    run_inference(model,train_ds,low_res_dir,out_folder,datetime.date(2004,1,1),datetime.date(2023,12,31))
    print("✅ 완료!")

if __name__=='__main__':
    main()


In [ ]:
import numpy as np
import torch
import rasterio
from netCDF4 import Dataset as ncDataset

# ─────────────────────────────────────────────────────────
# 좌표 방향 통일 함수들
# ─────────────────────────────────────────────────────────

def load_dem_with_correct_orientation(dem_file, target_shape):
    """DEM을 올바른 방향으로 로딩"""
    with rasterio.open(dem_file) as src:
        # DEM 데이터 로드
        dem = src.read(1).astype(np.float32)
        
        # GeoTIFF 좌표 정보 확인
        transform = src.transform
        
        # Y축 방향 확인 (음수면 북쪽이 위쪽)
        if transform[4] < 0:  # 일반적인 GeoTIFF는 음수
            print("DEM: 북쪽이 위쪽 (일반적)")
            # NetCDF 형식에 맞게 뒤집기 (남쪽이 위쪽으로)
            dem = np.flipud(dem)
            print("DEM을 남쪽-위쪽 방향으로 뒤집었습니다")
        else:
            print("DEM: 남쪽이 위쪽")
    
    # 목표 크기로 리샘플링
    dem_tensor = torch.tensor(dem, dtype=torch.float32).unsqueeze(0).unsqueeze(0)
    dem_resampled = torch.nn.functional.interpolate(
        dem_tensor, size=target_shape, mode='bilinear', align_corners=False
    ).squeeze().numpy()
    
    return dem_resampled

def check_netcdf_orientation(nc_file, var_name):
    """NetCDF 파일의 좌표 방향 확인"""
    with ncDataset(nc_file, 'r') as nc:
        if 'lat' in nc.variables:
            lat = nc.variables['lat'][:]
        elif 'latitude' in nc.variables:
            lat = nc.variables['latitude'][:]
        else:
            print("위도 변수를 찾을 수 없습니다")
            return None
            
        # 위도 방향 확인
        if lat[0] < lat[-1]:
            print(f"{nc_file}: 남쪽에서 북쪽으로 (오름차순)")
            return "south_to_north"
        else:
            print(f"{nc_file}: 북쪽에서 남쪽으로 (내림차순)")
            return "north_to_south"

def load_nc_with_correct_orientation(nc_file, var_name, target_shape=None):
    """NetCDF를 올바른 방향으로 로딩"""
    with ncDataset(nc_file, 'r') as nc:
        data = nc.variables[var_name][:].astype(np.float32)
        data = data.squeeze() if data.ndim == 3 else data
        
        # 위도 정보 확인
        if 'lat' in nc.variables:
            lat = nc.variables['lat'][:]
        elif 'latitude' in nc.variables:
            lat = nc.variables['latitude'][:]
        else:
            lat = None
            
        # 위도가 내림차순이면 뒤집기 (북쪽→남쪽을 남쪽→북쪽으로)
        if lat is not None and lat[0] > lat[-1]:
            print(f"NetCDF 데이터를 남쪽-위쪽 방향으로 뒤집었습니다")
            data = np.flipud(data)
    
    # 리샘플링이 필요한 경우
    if target_shape and data.shape != target_shape:
        data_tensor = torch.tensor(data, dtype=torch.float32).unsqueeze(0).unsqueeze(0)
        data = torch.nn.functional.interpolate(
            data_tensor, size=target_shape, mode='bilinear', align_corners=False
        ).squeeze().numpy()
    
    return data

def save_netcdf_with_correct_coords(data, date_str, output_dir, lat_start=33.1, lat_end=33.7, 
                                   lon_start=126.1, lon_end=127.1):
    """올바른 좌표로 NetCDF 저장"""
    import os
    from netCDF4 import Dataset as ncDataset
    
    os.makedirs(output_dir, exist_ok=True)
    outfn = os.path.join(output_dir, f"{date_str}_Downscaled_Precipitation.nc")
    
    n_lat, n_lon = data.shape
    
    # 좌표 생성 (남쪽에서 북쪽으로 오름차순)
    lat_vals = np.linspace(lat_start, lat_end, n_lat, dtype='f4')
    lon_vals = np.linspace(lon_start, lon_end, n_lon, dtype='f4')
    
    print(f"저장할 좌표 - Lat: {lat_vals[0]:.3f}~{lat_vals[-1]:.3f}, Lon: {lon_vals[0]:.3f}~{lon_vals[-1]:.3f}")
    
    time_unit = f"days since {date_str[:4]}-{date_str[4:6]}-{date_str[6:]} 00:00:00"
    
    with ncDataset(outfn, 'w', format='NETCDF4') as ds:
        # 차원 생성
        ds.createDimension('time', 1)
        ds.createDimension('lat', n_lat)
        ds.createDimension('lon', n_lon)
        
        # 시간 변수
        tv = ds.createVariable('time', 'f4', ('time',))
        tv[:] = [0.0]
        tv.units = time_unit
        tv.calendar = 'gregorian'
        
        # 위도 변수 (오름차순)
        lv = ds.createVariable('lat', 'f4', ('lat',))
        lv[:] = lat_vals
        lv.units = 'degrees_north'
        lv.standard_name = 'latitude'
        lv.axis = 'Y'
        lv.long_name = 'latitude'
        
        # 경도 변수 (오름차순)
        lov = ds.createVariable('lon', 'f4', ('lon',))
        lov[:] = lon_vals
        lov.units = 'degrees_east'
        lov.standard_name = 'longitude'
        lov.axis = 'X'
        lov.long_name = 'longitude'
        
        # 강수량 변수
        p = ds.createVariable('precipitation', 'f4', ('time', 'lat', 'lon'),
                              zlib=True, complevel=4, fill_value=-9999.0)
        p[0, :, :] = data.astype('f4')
        p.units = 'mm/day'
        p.long_name = 'Downscaled Precipitation'
        p.coordinates = 'time lat lon'
        p.grid_mapping = 'crs'
        
        # CRS 정보
        crs = ds.createVariable('crs', 'i4')
        crs.grid_mapping_name = 'latitude_longitude'
        crs.epsg_code = 'EPSG:4326'
        crs.semi_major_axis = 6378137.0
        crs.inverse_flattening = 298.257223563
        
        # 전역 속성
        ds.Conventions = 'CF-1.8'
        ds.title = f"Downscaled Precipitation for {date_str}"
        ds.history = f"Created on {datetime.datetime.now():%Y-%m-%d %H:%M:%S}"
        ds.source = "AI Downscaling Model"
    
    print(f"NetCDF 저장 완료: {outfn}")

# ─────────────────────────────────────────────────────────
# Dataset 클래스 수정 버전
# ─────────────────────────────────────────────────────────

class RainfallDataset_Fixed(Dataset):
    def __init__(self, low_res_folder, high_res_folder, dem_file, start_date, end_date):
        # 기본 초기화는 동일...
        
        # DEM 로딩 - 좌표 방향 수정
        print("DEM 데이터 로딩 및 방향 확인 중...")
        self.dem_data = load_dem_with_correct_orientation(dem_file, (61, 101))
        
        # 첫 번째 파일로 좌표 방향 확인
        if self.valid_data:
            sample_low = os.path.join(low_res_folder, self.valid_data[0].strftime('%Y%m%d') + '.nc4')
            sample_high = os.path.join(high_res_folder, self.valid_data[0].strftime('%Y%m%d') + '_Cokriging_Precipitation.nc')
            
            print("파일 좌표 방향 확인:")
            check_netcdf_orientation(sample_low, 'precipitation')
            check_netcdf_orientation(sample_high, 'precipitation')
    
    def __getitem__(self, idx):
        date = self.valid_data[idx]
        low_file = os.path.join(self.low_res_folder, date.strftime('%Y%m%d') + '.nc4')
        high_file = os.path.join(self.high_res_folder, date.strftime('%Y%m%d') + '_Cokriging_Precipitation.nc')
        
        try:
            # 좌표 방향을 고려한 데이터 로딩
            low_arr = load_nc_with_correct_orientation(low_file, 'precipitation', (61, 101))
            high_arr = load_nc_with_correct_orientation(high_file, 'precipitation')
            
            # 정규화
            low_arr = self._normalize(low_arr)
            dem_arr = self._normalize(self.dem_data)
            
            # 텐서 변환
            low_t = torch.tensor(low_arr, dtype=torch.float32).unsqueeze(0)
            dem_t = torch.tensor(dem_arr, dtype=torch.float32).unsqueeze(0)
            high_t = torch.tensor(high_arr, dtype=torch.float32).unsqueeze(0)
            input_t = torch.cat([low_t, dem_t], dim=0)
            
            return input_t, high_t
            
        except Exception as e:
            print(f"데이터 로딩 오류 ({date}): {e}")
            # 오류 처리...
            
    def _normalize(self, arr):
        mean = np.mean(arr)
        std = np.std(arr)
        if std < 1e-8:
            return arr - mean
        return (arr - mean) / (std + 1e-8)

# ─────────────────────────────────────────────────────────
# 추론 함수 수정 버전
# ─────────────────────────────────────────────────────────

def run_inference_with_correct_coords(model, train_ds, low_res_dir, output_dir, start_date, end_date):
    """좌표 방향을 올바르게 처리하는 추론 함수"""
    device = next(model.parameters()).device
    model.eval()
    
    # DEM 정규화 (train_ds에서 가져오기)
    dem_norm = train_ds.dem_data  # 이미 정규화된 상태
    
    dates = [start_date + datetime.timedelta(days=i) 
             for i in range((end_date - start_date).days + 1)]
    
    print(f"추론 시작: {len(dates)}일 처리")
    
    for date in tqdm(dates, desc="Inference"):
        lf = os.path.join(low_res_dir, date.strftime('%Y%m%d') + '.nc4')
        
        if not os.path.exists(lf):
            continue
            
        try:
            # 좌표 방향 고려한 GPM 데이터 로딩
            low_arr = load_nc_with_correct_orientation(lf, 'precipitation', (61, 101))
            low_norm = train_ds._normalize(low_arr)
            
            # 입력 텐서 생성
            x = torch.tensor(np.stack([low_norm, dem_norm], axis=0), 
                           dtype=torch.float32).unsqueeze(0).to(device)
            
            # 예측
            with torch.no_grad():
                pred = model(x, (61, 101)).cpu().squeeze().numpy()
            
            # 음수 제거
            pred = np.clip(pred, 0, None)
            
            # 올바른 좌표로 NetCDF 저장
            save_netcdf_with_correct_coords(pred, date.strftime('%Y%m%d'), output_dir)
            
        except Exception as e:
            print(f"오류 발생 ({date.strftime('%Y%m%d')}): {e}")
            continue
    
    print("✅ 추론 완료!")

# ─────────────────────────────────────────────────────────
# 좌표 확인 유틸리티
# ─────────────────────────────────────────────────────────

def debug_coordinate_directions(dem_file, sample_nc_file):
    """좌표 방향 디버깅 함수"""
    print("=== 좌표 방향 디버깅 ===")
    
    # DEM 좌표 확인
    print("\n1. DEM 파일 좌표:")
    with rasterio.open(dem_file) as src:
        transform = src.transform
        print(f"   Transform: {transform}")
        print(f"   Y direction: {'북쪽→남쪽' if transform[4] < 0 else '남쪽→북쪽'}")
    
    # NetCDF 좌표 확인
    print("\n2. NetCDF 파일 좌표:")
    check_netcdf_orientation(sample_nc_file, 'precipitation')
    
    print("\n=== 해결책 ===")
    print("DEM과 NetCDF의 Y축 방향이 다르면 데이터가 뒤집혀 보입니다.")
    print("위 함수들을 사용하여 좌표 방향을 통일하세요.")

if __name__ == "__main__":
    # 좌표 방향 확인 예시
    dem_file = r"/mnt/c/Yeonji/2025.01.Drought/DEM(merge).tif"
    sample_nc = r"/mnt/c/Yeonji/GPM/Input/20040101.nc4"
    
    debug_coordinate_directions(dem_file, sample_nc)

In [ ]:
import os
import random
import datetime
import logging
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
from netCDF4 import Dataset as ncDataset
from tqdm import tqdm
from sklearn.metrics import mean_squared_error, mean_absolute_error
import rasterio
from rasterio.enums import Resampling

# ─────────────────────────────────────────────────────────
# 0) 전역 설정
# ─────────────────────────────────────────────────────────
NUM_WORKERS = 2
PIN_MEMORY = True

# 제주 영역: 0.01° 해상도 (목표 격자)
JEJU_LAT_START, JEJU_LAT_END = 33.1, 33.7  # 61 셀
JEJU_LON_START, JEJU_LON_END = 126.1, 127.1
RESOLUTION = 0.01
LAT_VALS = np.arange(JEJU_LAT_START, JEJU_LAT_END + RESOLUTION/2, RESOLUTION, dtype='f4')
LON_VALS = np.arange(JEJU_LON_START, JEJU_LON_END + RESOLUTION/2, RESOLUTION, dtype='f4')

print(f"Target grid size: {LAT_VALS.size} x {LON_VALS.size}")
print(f"Latitude range: {LAT_VALS[0]:.3f} ~ {LAT_VALS[-1]:.3f} (South to North)")
print(f"Longitude range: {LON_VALS[0]:.3f} ~ {LON_VALS[-1]:.3f}")

# ─────────────────────────────────────────────────────────
# 1) 좌표 방향 수정된 GeoTIFF DEM 로딩 함수
# ─────────────────────────────────────────────────────────
def load_dem_geotiff_fixed(dem_path, target_shape):
    """
    GeoTIFF DEM 파일을 좌표 방향 수정하여 target_shape로 리샘플링
    """
    print(f"📍 DEM GeoTIFF 로딩: {dem_path}")
    
    with rasterio.open(dem_path) as src:
        print(f"   원본 크기: {src.height} x {src.width}")
        print(f"   좌표계: {src.crs}")
        print(f"   Transform: {src.transform}")
        
        # Transform에서 Y 방향 확인
        y_direction = src.transform[4]  # transform[4]가 픽셀 크기의 Y 방향
        print(f"   Y 방향: {'북쪽→남쪽' if y_direction < 0 else '남쪽→북쪽'}")
        
        # 데이터 읽기 및 리샘플링
        data = src.read(
            1,
            out_shape=target_shape,
            resampling=Resampling.bilinear
        ).astype(np.float32)
        
        # Y 방향이 북쪽→남쪽이면 뒤집기 (일반적인 GeoTIFF 형태)
        if y_direction < 0:
            print("   ⚠️  DEM Y축이 북쪽→남쪽입니다. 남쪽→북쪽으로 뒤집는 중...")
            data = np.flipud(data)
            print("   ✅ DEM 좌표 방향 수정 완료")
        else:
            print("   ✅ DEM Y축이 이미 남쪽→북쪽입니다")
    
    print(f"   최종 크기: {data.shape}")
    return data

# ─────────────────────────────────────────────────────────
# 2) 좌표 방향 수정된 NetCDF 로딩 함수
# ─────────────────────────────────────────────────────────
def load_nc_with_coord_check(path, var, target_size=None):
    """NetCDF 파일을 좌표 방향 확인하여 로딩"""
    with ncDataset(path) as nc:
        arr = nc.variables[var][:].astype(np.float32)
        arr = arr.squeeze() if arr.ndim == 3 else arr
        
        # 위도 정보 확인
        lat_var = None
        if 'lat' in nc.variables:
            lat_var = nc.variables['lat'][:]
        elif 'latitude' in nc.variables:
            lat_var = nc.variables['latitude'][:]
        
        # 위도 방향 확인 및 수정
        if lat_var is not None:
            if lat_var[0] > lat_var[-1]:  # 북쪽→남쪽 (내림차순)
                # 남쪽→북쪽으로 뒤집기
                arr = np.flipud(arr)
        
        # 리샘플링 (필요한 경우)
        if target_size and arr.shape != target_size:
            arr_tensor = torch.tensor(arr, dtype=torch.float32).unsqueeze(0).unsqueeze(0)
            arr = torch.nn.functional.interpolate(
                arr_tensor, size=target_size, mode='bilinear', align_corners=False
            ).squeeze().numpy()
        
        return arr

# ─────────────────────────────────────────────────────────
# 3) NetCDF 저장 함수 (좌표 정보 개선)
# ─────────────────────────────────────────────────────────
def save_to_netcdf(data, date_str, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    outfn = os.path.join(output_dir, f"{date_str}_Downscaled_Precipitation.nc")
    n_lat, n_lon = LAT_VALS.size, LON_VALS.size
    time_unit = f"days since {date_str[:4]}-{date_str[4:6]}-{date_str[6:]} 00:00:00"
    
    with ncDataset(outfn, 'w', format='NETCDF4') as ds:
        # 차원 생성
        ds.createDimension('time', 1)
        ds.createDimension('lat', n_lat)
        ds.createDimension('lon', n_lon)
        
        # 시간 변수
        tv = ds.createVariable('time', 'f4', ('time',))
        tv[:] = [0.0]
        tv.units = time_unit
        tv.calendar = 'gregorian'
        
        # 위도 변수 (남쪽→북쪽 오름차순)
        lv = ds.createVariable('lat', 'f4', ('lat',))
        lv[:] = LAT_VALS
        lv.units = 'degrees_north'
        lv.standard_name = 'latitude'
        lv.axis = 'Y'
        lv.long_name = 'latitude'
        
        # 경도 변수
        lov = ds.createVariable('lon', 'f4', ('lon',))
        lov[:] = LON_VALS
        lov.units = 'degrees_east'
        lov.standard_name = 'longitude'
        lov.axis = 'X'
        lov.long_name = 'longitude'
        
        # 강수량 변수
        p = ds.createVariable('precipitation','f4',('time','lat','lon'),
                              zlib=True, complevel=4, fill_value=-9999.0)
        p[0,:,:] = data.astype('f4')
        p.units = 'mm/day'
        p.long_name = 'Downscaled Precipitation'
        p.coordinates = 'time lat lon'
        p.grid_mapping = 'crs'
        
        # CRS 정보
        crs = ds.createVariable('crs','i4')
        crs.grid_mapping_name = 'latitude_longitude'
        crs.epsg_code = 'EPSG:4326'
        crs.semi_major_axis = 6378137.0
        crs.inverse_flattening = 298.257223563
        
        # 전역 속성
        ds.Conventions = 'CF-1.8'
        ds.title = f"Downscaled Precipitation for {date_str}"
        ds.history = f"Created on {datetime.datetime.now():%Y-%m-%d %H:%M:%S}"
        ds.note = "Coordinates: lat/lon in ascending order (South to North, West to East)"

# ─────────────────────────────────────────────────────────
# 4) 좌표 방향 수정된 Dataset 클래스
# ─────────────────────────────────────────────────────────
class RainfallDataset(Dataset):
    def __init__(self, low_res_folder, high_res_folder, dem_file,
                 start_date, end_date, normalization_stats=None):
        self.low = low_res_folder
        self.high = high_res_folder
        self.dates = [start_date + datetime.timedelta(days=i)
                      for i in range((end_date - start_date).days + 1)]
        
        # 유효한 파일 쌍 수집
        self.paths = []
        for d in self.dates:
            f1 = os.path.join(self.low, d.strftime('%Y%m%d') + '.nc4')
            f2 = os.path.join(self.high, d.strftime('%Y%m%d') + '_Cokriging_Precipitation.nc')
            if os.path.exists(f1) and os.path.exists(f2):
                self.paths.append((f1,f2))
        
        if not self.paths:
            raise ValueError("유효한 데이터가 없습니다.")
        
        print(f"유효한 데이터 쌍: {len(self.paths)}개")
        
        # 좌표 방향 확인
        print(f"\n=== 좌표 방향 확인 ===")
        self._check_coordinate_directions()
        
        # DEM 로딩 (좌표 방향 수정)
        print(f"\n=== DEM 처리 ===")
        self.dem = load_dem_geotiff_fixed(
            dem_file,
            target_shape=(LAT_VALS.size, LON_VALS.size)
        )
        print(f"DEM 최종 크기: {self.dem.shape}")
        
        # 정규화 통계
        self.stats = normalization_stats or self._calc_stats()
    
    def _check_coordinate_directions(self):
        """샘플 파일들의 좌표 방향 확인"""
        sample_files = [
            (self.paths[0][0], "GPM"),
            (self.paths[0][1], "Co-kriging")
        ]
        
        for file_path, data_type in sample_files:
            try:
                with ncDataset(file_path) as nc:
                    if 'lat' in nc.variables:
                        lat = nc.variables['lat'][:]
                    elif 'latitude' in nc.variables:
                        lat = nc.variables['latitude'][:]
                    else:
                        print(f"  {data_type}: 위도 변수 없음")
                        continue
                        
                    if lat[0] < lat[-1]:
                        print(f"  {data_type}: 남쪽→북쪽 (오름차순) ✅")
                    else:
                        print(f"  {data_type}: 북쪽→남쪽 (내림차순) - 수정됨")
            except Exception as e:
                print(f"  {data_type}: 좌표 확인 오류 - {e}")
    
    def __len__(self):
        return len(self.paths)
    
    def __getitem__(self, idx):
        low_f, high_f = self.paths[idx]
        
        # 좌표 방향 확인하여 로딩
        low_arr = load_nc_with_coord_check(low_f, 'precipitation', (LAT_VALS.size, LON_VALS.size))
        high_arr = load_nc_with_coord_check(high_f, 'precipitation')
        
        # 정규화
        low_norm = (low_arr - self.stats['low_res']['mean']) / (self.stats['low_res']['std'] + 1e-8)
        high_norm = (high_arr - self.stats['high_res']['mean']) / (self.stats['high_res']['std'] + 1e-8)
        dem_norm = (self.dem - self.stats['dem']['mean']) / (self.stats['dem']['std'] + 1e-8)
        
        x = torch.tensor(np.stack([low_norm, dem_norm], axis=0), dtype=torch.float32)
        y = torch.tensor(high_norm, dtype=torch.float32).unsqueeze(0)
        return x, y
    
    def _calc_stats(self):
        print(f"\n=== 정규화 통계 계산 ===")
        lows, highs = [], []
        
        for i, (l, h) in enumerate(self.paths):
            if i % 100 == 0:
                print(f"  통계 계산 중: {i}/{len(self.paths)}")
            
            # 좌표 방향 수정하여 로딩
            low_data = load_nc_with_coord_check(l, 'precipitation', (LAT_VALS.size, LON_VALS.size))
            high_data = load_nc_with_coord_check(h, 'precipitation')
            
            lows.append(low_data)
            highs.append(high_data)
        
        lows = np.stack(lows)
        highs = np.stack(highs)
        
        stats = {
            'low_res':  {'mean': lows.mean(),  'std': lows.std()},
            'high_res': {'mean': highs.mean(), 'std': highs.std()},
            'dem':      {'mean': self.dem.mean(), 'std': self.dem.std()}
        }
        
        print("✅ 정규화 통계 완료:")
        for k, v in stats.items():
            print(f"  {k}: mean={v['mean']:.4f}, std={v['std']:.4f}")
            
        return stats

# ─────────────────────────────────────────────────────────
# 5) Generator 모델 정의
# ─────────────────────────────────────────────────────────
class Generator(nn.Module):
    def __init__(self, in_ch=2, n_layers=4, base_filters=32):
        super().__init__()
        layers = []
        c = in_ch
        for i in range(n_layers):
            f = base_filters * (2**i)
            layers += [nn.Conv2d(c,f,3,1,1,bias=False), nn.BatchNorm2d(f), nn.ReLU(True)]
            c = f
        self.encoder = nn.Sequential(*layers)
        self.final = nn.Conv2d(c,1,3,1,1)
    
    def forward(self, x):
        return self.final(self.encoder(x))

# ─────────────────────────────────────────────────────────
# 6) 학습 과정 표시 개선된 학습 함수
# ─────────────────────────────────────────────────────────
def train_final(train_ds, val_ds, test_ds, params, log_dir, epochs=50):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    print(f"\n{'='*60}")
    print(f"🚀 모델 학습 시작")
    print(f"{'='*60}")
    print(f"디바이스: {device}")
    print(f"에포크 수: {epochs}")
    print(f"배치 크기: {params['batch_size']}")
    print(f"학습률: {params['lr_g']:.2e}")
    print(f"손실 함수: {params['loss_function']}")
    
    logging.info(f"Using device: {device}")
    
    # 모델 초기화
    G = Generator(2, params['n_layers_g'], params['n_filters_base_g']).to(device)
    optimG = optim.AdamW(G.parameters(), lr=params['lr_g'], betas=(0.5,0.999))
    criterion = nn.L1Loss() if params['loss_function']=='L1' else nn.MSELoss()
    scaler = GradScaler(enabled=torch.cuda.is_available())
    
    # 데이터로더
    tl = DataLoader(train_ds, batch_size=params['batch_size'], shuffle=True, 
                   pin_memory=PIN_MEMORY, num_workers=NUM_WORKERS)
    vl = DataLoader(val_ds, batch_size=params['batch_size'], shuffle=False,
                   pin_memory=PIN_MEMORY, num_workers=NUM_WORKERS)
    
    print(f"배치 수 - Train: {len(tl)}, Val: {len(vl)}")
    
    best, cnt = 1e9, 0
    train_losses, val_losses = [], []
    
    print(f"\n{'='*70}")
    print(f"{'Epoch':>5} {'Train Loss':>12} {'Val Loss':>10} {'Best Val':>10} {'Status':>12}")
    print(f"{'='*70}")
    
    for ep in range(1, epochs+1):
        # 학습 모드
        G.train()
        train_loss = 0
        batch_count = 0
        
        # 에포크별 진행바
        pbar = tqdm(tl, desc=f"Epoch {ep:2d}/{epochs}", leave=False, ncols=80)
        
        for x, y in pbar:
            x, y = x.to(device), y.to(device)
            optimG.zero_grad()
            
            with autocast(device_type=device.type, enabled=torch.cuda.is_available()):
                loss = criterion(G(x), y) * params['content_weight']
            
            scaler.scale(loss).backward()
            scaler.step(optimG)
            scaler.update()
            train_loss += loss.item()
            batch_count += 1
            
            # 진행바 업데이트
            pbar.set_postfix({'Loss': f"{loss.item():.4f}"})
        
        train_loss /= max(batch_count, 1)
        train_losses.append(train_loss)
        
        # 검증 모드
        G.eval()
        val_loss = 0
        val_batch_count = 0
        
        with torch.no_grad():
            for x, y in vl:
                val_loss += criterion(G(x.to(device)), y.to(device)).item()
                val_batch_count += 1
        
        val_loss /= max(val_batch_count, 1)
        val_losses.append(val_loss)
        
        # 결과 출력
        status = ""
        if val_loss < best:
            best, cnt = val_loss, 0
            torch.save(G.state_dict(), os.path.join(log_dir,'best_generator.pth'))
            status = "💾 SAVED"
        else:
            cnt += 1
            status = f"⏳ {cnt}/15"
        
        print(f"{ep:5d} {train_loss:12.6f} {val_loss:10.6f} {best:10.6f} {status:>12}")
        
        # 로그 기록
        logging.info(f"Epoch {ep}, Train {train_loss:.6f}, Val {val_loss:.6f}")
        
        # Early stopping
        if cnt >= 15:
            print(f"\n⏹️  Early stopping at epoch {ep} (no improvement for 15 epochs)")
            break
        
        # 5 에포크마다 진행 상황 요약
        if ep % 5 == 0:
            recent_train = np.mean(train_losses[-5:])
            recent_val = np.mean(val_losses[-5:])
            print(f"    📊 최근 5 에포크 평균 - Train: {recent_train:.6f}, Val: {recent_val:.6f}")
    
    print(f"{'='*70}")
    print(f"🎯 학습 완료! 최고 검증 손실: {best:.6f}")
    
    # 테스트 평가
    print(f"\n=== 테스트 평가 ===")
    G.load_state_dict(torch.load(os.path.join(log_dir,'best_generator.pth')))
    tle = DataLoader(test_ds, batch_size=1, shuffle=False)
    ms, ma = [], []
    
    G.eval()
    with torch.no_grad():
        for i, (x, y) in enumerate(tqdm(tle, desc="Testing", ncols=80)):
            x, y = x.to(device), y.to(device)
            p = G(x)
            y_, p_ = y.cpu().numpy().flatten(), p.cpu().numpy().flatten()
            ms.append(mean_squared_error(y_, p_))
            ma.append(mean_absolute_error(y_, p_))
            
            # 첫 5개 샘플 통계 출력
            if i < 5:
                print(f"  샘플 {i+1}: MSE={ms[-1]:.6f}, MAE={ma[-1]:.6f}")
    
    final_mse, final_mae = np.mean(ms), np.mean(ma)
    print(f"\n📈 최종 테스트 결과:")
    print(f"  평균 MSE: {final_mse:.6f}")
    print(f"  평균 MAE: {final_mae:.6f}")
    
    return G, final_mse, final_mae

# ─────────────────────────────────────────────────────────
# 7) 좌표 수정된 추론 함수
# ─────────────────────────────────────────────────────────
def run_inference(model, train_ds, low_res_dir, output_dir, start_date, end_date):
    print(f"\n{'='*60}")
    print(f"🔮 추론 및 NetCDF 생성")
    print(f"{'='*60}")
    
    device = next(model.parameters()).device
    model.eval()
    dem_norm = (train_ds.dem - train_ds.stats['dem']['mean']) / (train_ds.stats['dem']['std'] + 1e-8)
    dates = [start_date + datetime.timedelta(days=i) for i in range((end_date-start_date).days+1)]
    
    print(f"처리할 날짜 수: {len(dates)}일")
    print(f"출력 디렉토리: {output_dir}")
    
    processed_count = 0
    
    for date in tqdm(dates, desc="Downscaling", ncols=80):
        lf = os.path.join(low_res_dir, date.strftime('%Y%m%d') + '.nc4')
        if not os.path.exists(lf):
            continue
        
        try:
            # 좌표 방향 수정하여 GPM 로딩
            arr = load_nc_with_coord_check(lf, 'precipitation', (LAT_VALS.size, LON_VALS.size))
            
            # 정규화
            low_norm = (arr - train_ds.stats['low_res']['mean']) / (train_ds.stats['low_res']['std'] + 1e-8)
            
            # 예측
            x = torch.tensor(np.stack([low_norm, dem_norm], axis=0), dtype=torch.float32).unsqueeze(0).to(device)
            with torch.no_grad():
                pred = model(x).cpu().squeeze().numpy()
            
            # 역정규화
            pred_den = pred * train_ds.stats['high_res']['std'] + train_ds.stats['high_res']['mean']
            pred_den = np.clip(pred_den, 0, None)
            
            # NetCDF 저장
            save_to_netcdf(pred_den, date.strftime('%Y%m%d'), output_dir)
            processed_count += 1
            
            # 진행 상황 출력
            if processed_count % 100 == 0:
                print(f"  ✅ {processed_count}개 파일 처리 완료")
                
        except Exception as e:
            print(f"  ❌ 오류 ({date.strftime('%Y%m%d')}): {e}")
            continue
    
    print(f"\n🎉 추론 완료!")
    print(f"  총 처리된 파일: {processed_count}개")

# ─────────────────────────────────────────────────────────
# 8) 메인 실행
# ─────────────────────────────────────────────────────────
def main():
    print("🌧️ 강수량 다운스케일링 (GeoTIFF DEM 버전)")
    print(f"실행 시간: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    
    random.seed(42)
    np.random.seed(42)
    torch.manual_seed(42)
    
    # 로그 디렉토리
    run_dt = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
    log_dir = f"/mnt/c/Yeonji/2025.01.Drought/2004/{run_dt}"
    os.makedirs(log_dir, exist_ok=True)
    
    logging.basicConfig(level=logging.INFO, filename=os.path.join(log_dir,'train.log'),
                       format='%(asctime)s - %(message)s')
    
    # 디바이스 정보
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"사용 디바이스: {device}")
    
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
        print(f"GPU 메모리: {torch.cuda.get_device_properties(0).total_memory // 1024**3} GB")
    
    # 경로 설정
    low_res_dir = r"/mnt/c/Yeonji/GPM/Input"
    high_res_dir = r"/mnt/c/Yeonji/2025.01.Drought/2004/1.CoKriging(Daily)"
    dem_file = r"/mnt/c/Yeonji/2025.01.Drought/DEM(merge).tif"  # GeoTIFF
    
    print(f"\n📂 경로 확인:")
    print(f"  GPM 입력: {low_res_dir}")
    print(f"  Co-kriging: {high_res_dir}")
    print(f"  DEM 파일: {dem_file}")
    print(f"  로그 디렉토리: {log_dir}")
    
    # 경로 존재 확인
    for path_name, path in [("GPM", low_res_dir), ("Co-kriging", high_res_dir)]:
        if not os.path.exists(path):
            raise FileNotFoundError(f"{path_name} 디렉토리가 존재하지 않습니다: {path}")
        else:
            print(f"  ✅ {path_name} 디렉토리 확인됨")
    
    if not os.path.exists(dem_file):
        raise FileNotFoundError(f"DEM 파일이 존재하지 않습니다: {dem_file}")
    else:
        print(f"  ✅ DEM 파일 확인됨")
    
    print(f"\n{'='*60}")
    print(f"🔧 데이터셋 로딩 및 좌표 방향 수정")
    print(f"{'='*60}")
    
    # 데이터셋 생성
    train_ds = RainfallDataset(low_res_dir, high_res_dir, dem_file,
                              datetime.date(2004,1,1), datetime.date(2018,12,31))
    val_ds = RainfallDataset(low_res_dir, high_res_dir, dem_file,
                            datetime.date(2019,1,1), datetime.date(2021,12,31),
                            normalization_stats=train_ds.stats)
    test_ds = RainfallDataset(low_res_dir, high_res_dir, dem_file,
                             datetime.date(2022,1,1), datetime.date(2023,12,31),
                             normalization_stats=train_ds.stats)
    
    print(f"\n📊 데이터셋 크기:")
    print(f"  Train: {len(train_ds):,}개")
    print(f"  Validation: {len(val_ds):,}개")
    print(f"  Test: {len(test_ds):,}개")
    print(f"  총합: {len(train_ds) + len(val_ds) + len(test_ds):,}개")
    
    # 하이퍼파라미터
    best_params = {
        'batch_size': 32,
        'lr_g': 1.6237e-05,
        'content_weight': 137.9359,
        'n_layers_g': 4,
        'n_filters_base_g': 32,
        'loss_function': 'L1'
    }
    
    print(f"\n⚙️ 하이퍼파라미터:")
    for key, value in best_params.items():
        if isinstance(value, float):
            print(f"  {key}: {value:.2e}" if value < 0.001 else f"  {key}: {value:.4f}")
        else:
            print(f"  {key}: {value}")
    
    print(f"\n{'='*60}")
    print(f"🚀 모델 학습")
    print(f"{'='*60}")
    
    # 모델 학습
    model, mse, mae = train_final(train_ds, val_ds, test_ds, best_params, log_dir, epochs=50)
    
    print(f"\n🎯 최종 성능:")
    print(f"  Test MSE: {mse:.6f}")
    print(f"  Test MAE: {mae:.6f}")
    
    # 모델 정보 저장
    model_info_file = os.path.join(log_dir, 'model_info.txt')
    with open(model_info_file, 'w') as f:
        f.write(f"모델 학습 완료 시간: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"디바이스: {device}\n")
        f.write(f"데이터셋 크기: Train={len(train_ds)}, Val={len(val_ds)}, Test={len(test_ds)}\n")
        f.write(f"최종 성능: MSE={mse:.6f}, MAE={mae:.6f}\n")
        f.write(f"DEM 파일: {dem_file} (GeoTIFF)\n")
        f.write(f"좌표 수정: Y축 방향 통일 (남쪽→북쪽)\n")
        f.write(f"하이퍼파라미터:\n")
        for k, v in best_params.items():
            f.write(f"  {k}: {v}\n")
        f.write(f"좌표 설정:\n")
        f.write(f"  위도 범위: {JEJU_LAT_START}~{JEJU_LAT_END} (남쪽→북쪽)\n")
        f.write(f"  경도 범위: {JEJU_LON_START}~{JEJU_LON_END}\n")
        f.write(f"  격자 크기: {LAT_VALS.size}×{LON_VALS.size}\n")
    
    print(f"📋 모델 정보가 저장되었습니다: {model_info_file}")
    
    # 추론
    print(f"\n{'='*60}")
    print(f"🔮 추론 및 NetCDF 생성")
    print(f"{'='*60}")
    
    out_folder = r"/mnt/c/Yeonji/2025.01.Drought/2004/0713(3)"
    run_inference(model, train_ds, low_res_dir, out_folder,
                  datetime.date(2004,1,1), datetime.date(2023,12,31))
    
    print(f"\n{'='*60}")
    print(f"🎉 전체 파이프라인 완료!")
    print(f"{'='*60}")
    print(f"📁 결과 파일:")
    print(f"  모델: {os.path.join(log_dir, 'best_generator.pth')}")
    print(f"  로그: {os.path.join(log_dir, 'train.log')}")
    print(f"  모델 정보: {model_info_file}")
    print(f"  다운스케일링 결과: {out_folder}")
    print(f"⏰ 완료 시간: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    
    # 결과 검증
    print(f"\n🔍 결과 검증:")
    try:
        # 생성된 파일 수 확인
        nc_files = [f for f in os.listdir(out_folder) if f.endswith('.nc')]
        print(f"  생성된 NetCDF 파일 수: {len(nc_files)}개")
        
        if nc_files:
            # 첫 번째 파일 상세 확인
            sample_file = os.path.join(out_folder, nc_files[0])
            with ncDataset(sample_file, 'r') as nc:
                lat = nc.variables['lat'][:]
                lon = nc.variables['lon'][:]
                precip = nc.variables['precipitation'][:]
                
                print(f"  샘플 파일: {nc_files[0]}")
                print(f"    위도 범위: {lat[0]:.3f}~{lat[-1]:.3f} ({'오름차순 ✅' if lat[0] < lat[-1] else '내림차순 ⚠️'})")
                print(f"    경도 범위: {lon[0]:.3f}~{lon[-1]:.3f}")
                print(f"    강수량 범위: {precip.min():.3f}~{precip.max():.3f} mm/day")
                print(f"    데이터 크기: {precip.shape}")
                print(f"    양수 강수량 비율: {(precip > 0).sum() / precip.size * 100:.1f}%")
                
                # 좌표 방향 최종 확인
                if lat[0] < lat[-1]:
                    print(f"  ✅ 좌표 방향 정상: 남쪽→북쪽 (DEM과 일치)")
                else:
                    print(f"  ⚠️  좌표 방향 확인 필요: 북쪽→남쪽")
                    
                # 좌표 정확성 확인
                expected_lat_start = JEJU_LAT_START
                expected_lat_end = JEJU_LAT_END
                if abs(lat[0] - expected_lat_start) < 0.01 and abs(lat[-1] - expected_lat_end) < 0.01:
                    print(f"  ✅ 좌표 정확성: 예상 범위와 일치")
                else:
                    print(f"  ⚠️  좌표 정확성: 예상({expected_lat_start:.3f}~{expected_lat_end:.3f})과 다름")
        
        # 마지막 몇 개 파일도 확인
        if len(nc_files) > 1:
            last_file = os.path.join(out_folder, nc_files[-1])
            with ncDataset(last_file, 'r') as nc:
                last_precip = nc.variables['precipitation'][:]
                print(f"  마지막 파일: {nc_files[-1]}")
                print(f"    강수량 범위: {last_precip.min():.3f}~{last_precip.max():.3f} mm/day")
                
    except Exception as e:
        print(f"  ❌ 검증 중 오류: {e}")
    
    # 성능 요약
    print(f"\n📈 최종 성능 요약:")
    print(f"  🎯 Test MSE: {mse:.6f}")
    print(f"  🎯 Test MAE: {mae:.6f}")
    print(f"  📊 처리된 기간: 2004-2023 (20년)")
    print(f"  🗺️  해상도: 0.1° → 0.01° (10배 향상)")
    print(f"  📐 격자 크기: {LAT_VALS.size} × {LON_VALS.size}")
    print(f"  🏔️  DEM: GeoTIFF 형식, 좌표 방향 자동 수정")
    
    print(f"\n{'='*60}")
    print(f"🎊 모든 작업이 성공적으로 완료되었습니다!")
    print(f"{'='*60}")

if __name__ == '__main__':
    try:
        main()
    except Exception as e:
        print(f"\n❌ 실행 중 오류 발생: {e}")
        logging.error(f"메인 함수 실행 중 오류: {e}", exc_info=True)
        print(f"자세한 오류 정보는 로그 파일을 확인하세요.")
        raise

In [ ]:
import os
import random
import datetime
import logging
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
from netCDF4 import Dataset as ncDataset
from tqdm import tqdm
from sklearn.metrics import mean_squared_error, mean_absolute_error
import rasterio
import rasterio.windows
from rasterio.enums import Resampling

# ─────────────────────────────────────────────────────────
# 0) 전역 설정 (속도 최적화)
# ─────────────────────────────────────────────────────────
NUM_WORKERS = 4  # 2 -> 4로 증가 (CPU 코어 수에 따라 조정)
PIN_MEMORY = True

# 제주 영역: 0.01° 해상도 (목표 격자)
JEJU_LAT_START, JEJU_LAT_END = 33.1, 33.7  # 61 셀
JEJU_LON_START, JEJU_LON_END = 126.1, 127.1
RESOLUTION = 0.01
LAT_VALS = np.arange(JEJU_LAT_START, JEJU_LAT_END + RESOLUTION/2, RESOLUTION, dtype='f4')
LON_VALS = np.arange(JEJU_LON_START, JEJU_LON_END + RESOLUTION/2, RESOLUTION, dtype='f4')

print(f"Target grid size: {LAT_VALS.size} x {LON_VALS.size}")
print(f"Latitude range: {LAT_VALS[0]:.3f} ~ {LAT_VALS[-1]:.3f} (South to North)")
print(f"Longitude range: {LON_VALS[0]:.3f} ~ {LON_VALS[-1]:.3f}")

# CUDA 최적화 설정
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True  # 고정 입력 크기에 대한 최적화
    torch.backends.cudnn.deterministic = False  # 재현성보다 속도 우선

# ─────────────────────────────────────────────────────────
# 1) 좌표 범위 제한된 GeoTIFF DEM 로딩 함수
# ─────────────────────────────────────────────────────────
def load_dem_geotiff_fixed(dem_path, target_shape):
    """
    GeoTIFF DEM 파일을 좌표 방향 수정하고 목표 범위만 잘라서 target_shape로 리샘플링
    """
    print(f"📍 DEM GeoTIFF 로딩: {dem_path}")
    
    with rasterio.open(dem_path) as src:
        print(f"   원본 크기: {src.height} x {src.width}")
        print(f"   좌표계: {src.crs}")
        print(f"   Transform: {src.transform}")
        print(f"   경계: {src.bounds}")
        
        # Transform에서 Y 방향 확인
        y_direction = src.transform[4]  # transform[4]가 픽셀 크기의 Y 방향
        print(f"   Y 방향: {'북쪽→남쪽' if y_direction < 0 else '남쪽→북쪽'}")
        
        # 목표 좌표 범위 계산
        target_bounds = (JEJU_LON_START, JEJU_LAT_START, JEJU_LON_END, JEJU_LAT_END)
        print(f"   목표 범위: Lon {JEJU_LON_START}~{JEJU_LON_END}, Lat {JEJU_LAT_START}~{JEJU_LAT_END}")
        
        # 좌표계가 지리좌표계(위경도)인지 확인
        if src.crs.is_geographic:
            print("   ✅ 지리좌표계 (위경도) 확인됨")
            crop_bounds = target_bounds
        else:
            print(f"   ⚠️  투영좌표계 감지: {src.crs}")
            # 투영좌표계인 경우 지리좌표를 투영좌표로 변환
            from rasterio.warp import transform_bounds
            import pyproj
            
            try:
                # 지리좌표 -> 투영좌표 변환
                crop_bounds = transform_bounds(
                    'EPSG:4326',  # WGS84 (위경도)
                    src.crs,      # DEM의 좌표계
                    *target_bounds
                )
                print(f"   변환된 투영 범위: {[f'{x:.2f}' for x in crop_bounds]}")
            except Exception as e:
                print(f"   ❌ 좌표 변환 실패: {e}")
                print("   전체 DEM 사용")
                crop_bounds = src.bounds
        
        # 윈도우 계산 (DEM에서 목표 범위에 해당하는 픽셀 영역)
        try:
            from rasterio.windows import from_bounds
            window = from_bounds(*crop_bounds, src.transform)
            print(f"   윈도우: {window}")
            
            # 윈도우가 DEM 범위를 벗어나지 않도록 조정
            window = window.intersection(rasterio.windows.Window(0, 0, src.width, src.height))
            print(f"   조정된 윈도우: {window}")
            
            # 윈도우 영역의 데이터 읽기
            if window.width > 0 and window.height > 0:
                data = src.read(1, window=window).astype(np.float32)
                print(f"   크롭된 크기: {data.shape}")
            else:
                print("   ❌ 유효한 윈도우 없음, 전체 DEM 사용")
                data = src.read(1).astype(np.float32)
                
        except Exception as e:
            print(f"   ❌ 윈도우 계산 실패: {e}")
            print("   전체 DEM 사용")
            data = src.read(1).astype(np.float32)
        
        # Y 방향이 북쪽→남쪽이면 뒤집기 (일반적인 GeoTIFF 형태)
        if y_direction < 0:
            print("   ⚠️  DEM Y축이 북쪽→남쪽입니다. 남쪽→북쪽으로 뒤집는 중...")
            data = np.flipud(data).copy()  # Fix negative strides
            print("   ✅ DEM 좌표 방향 수정 완료")
        else:
            print("   ✅ DEM Y축이 이미 남쪽→북쪽입니다")
        
        # 목표 크기로 리샘플링
        if data.shape != target_shape:
            print(f"   리샘플링: {data.shape} -> {target_shape}")
            data_tensor = torch.tensor(data, dtype=torch.float32).unsqueeze(0).unsqueeze(0)
            data = torch.nn.functional.interpolate(
                data_tensor, size=target_shape, mode='bilinear', align_corners=False
            ).squeeze().numpy()
        else:
            print("   크기가 이미 일치함")
    
    print(f"   최종 크기: {data.shape}")
    return data

# ─────────────────────────────────────────────────────────
# 2) 좌표 방향 수정된 NetCDF 로딩 함수
# ─────────────────────────────────────────────────────────
def load_nc_with_coord_check(path, var, target_size=None):
    """NetCDF 파일을 좌표 방향 확인하여 로딩"""
    with ncDataset(path) as nc:
        arr = nc.variables[var][:].astype(np.float32)
        arr = arr.squeeze() if arr.ndim == 3 else arr
        
        # 위도 정보 확인
        lat_var = None
        if 'lat' in nc.variables:
            lat_var = nc.variables['lat'][:]
        elif 'latitude' in nc.variables:
            lat_var = nc.variables['latitude'][:]
        
        # 위도 방향 확인 및 수정
        if lat_var is not None:
            if lat_var[0] > lat_var[-1]:  # 북쪽→남쪽 (내림차순)
                # 남쪽→북쪽으로 뒤집기
                arr = np.flipud(arr)
        
        # 리샘플링 (필요한 경우)
        if target_size and arr.shape != target_size:
            arr_tensor = torch.tensor(arr, dtype=torch.float32).unsqueeze(0).unsqueeze(0)
            arr = torch.nn.functional.interpolate(
                arr_tensor, size=target_size, mode='bilinear', align_corners=False
            ).squeeze().numpy()
        
        return arr

# ─────────────────────────────────────────────────────────
# 3) NetCDF 저장 함수 (좌표 정보 개선)
# ─────────────────────────────────────────────────────────
def save_to_netcdf(data, date_str, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    outfn = os.path.join(output_dir, f"{date_str}_Downscaled_Precipitation.nc")
    n_lat, n_lon = LAT_VALS.size, LON_VALS.size
    time_unit = f"days since {date_str[:4]}-{date_str[4:6]}-{date_str[6:]} 00:00:00"
    
    with ncDataset(outfn, 'w', format='NETCDF4') as ds:
        # 차원 생성
        ds.createDimension('time', 1)
        ds.createDimension('lat', n_lat)
        ds.createDimension('lon', n_lon)
        
        # 시간 변수
        tv = ds.createVariable('time', 'f4', ('time',))
        tv[:] = [0.0]
        tv.units = time_unit
        tv.calendar = 'gregorian'
        
        # 위도 변수 (남쪽→북쪽 오름차순)
        lv = ds.createVariable('lat', 'f4', ('lat',))
        lv[:] = LAT_VALS
        lv.units = 'degrees_north'
        lv.standard_name = 'latitude'
        lv.axis = 'Y'
        lv.long_name = 'latitude'
        
        # 경도 변수
        lov = ds.createVariable('lon', 'f4', ('lon',))
        lov[:] = LON_VALS
        lov.units = 'degrees_east'
        lov.standard_name = 'longitude'
        lov.axis = 'X'
        lov.long_name = 'longitude'
        
        # 강수량 변수
        p = ds.createVariable('precipitation','f4',('time','lat','lon'),
                              zlib=True, complevel=4, fill_value=-9999.0)
        p[0,:,:] = data.astype('f4')
        p.units = 'mm/day'
        p.long_name = 'Downscaled Precipitation'
        p.coordinates = 'time lat lon'
        p.grid_mapping = 'crs'
        
        # CRS 정보
        crs = ds.createVariable('crs','i4')
        crs.grid_mapping_name = 'latitude_longitude'
        crs.epsg_code = 'EPSG:4326'
        crs.semi_major_axis = 6378137.0
        crs.inverse_flattening = 298.257223563
        
        # 전역 속성
        ds.Conventions = 'CF-1.8'
        ds.title = f"Downscaled Precipitation for {date_str}"
        ds.history = f"Created on {datetime.datetime.now():%Y-%m-%d %H:%M:%S}"
        ds.note = "Coordinates: lat/lon in ascending order (South to North, West to East)"

# ─────────────────────────────────────────────────────────
# 4) 좌표 방향 수정된 Dataset 클래스
# ─────────────────────────────────────────────────────────
class RainfallDataset(Dataset):
    def __init__(self, low_res_folder, high_res_folder, dem_file,
                 start_date, end_date, normalization_stats=None):
        self.low = low_res_folder
        self.high = high_res_folder
        self.dates = [start_date + datetime.timedelta(days=i)
                      for i in range((end_date - start_date).days + 1)]
        
        # 유효한 파일 쌍 수집
        self.paths = []
        for d in self.dates:
            f1 = os.path.join(self.low, d.strftime('%Y%m%d') + '.nc4')
            f2 = os.path.join(self.high, d.strftime('%Y%m%d') + '_Cokriging_Precipitation.nc')
            if os.path.exists(f1) and os.path.exists(f2):
                self.paths.append((f1,f2))
        
        if not self.paths:
            raise ValueError("유효한 데이터가 없습니다.")
        
        print(f"유효한 데이터 쌍: {len(self.paths)}개")
        
        # 좌표 방향 확인
        print(f"\n=== 좌표 방향 확인 ===")
        self._check_coordinate_directions()
        
        # DEM 로딩 (좌표 방향 수정)
        print(f"\n=== DEM 처리 ===")
        self.dem = load_dem_geotiff_fixed(
            dem_file,
            target_shape=(LAT_VALS.size, LON_VALS.size)
        )
        print(f"DEM 최종 크기: {self.dem.shape}")
        
        # 정규화 통계
        self.stats = normalization_stats or self._calc_stats()
    
    def _check_coordinate_directions(self):
        """샘플 파일들의 좌표 방향 확인"""
        sample_files = [
            (self.paths[0][0], "GPM"),
            (self.paths[0][1], "Co-kriging")
        ]
        
        for file_path, data_type in sample_files:
            try:
                with ncDataset(file_path) as nc:
                    if 'lat' in nc.variables:
                        lat = nc.variables['lat'][:]
                    elif 'latitude' in nc.variables:
                        lat = nc.variables['latitude'][:]
                    else:
                        print(f"  {data_type}: 위도 변수 없음")
                        continue
                        
                    if lat[0] < lat[-1]:
                        print(f"  {data_type}: 남쪽→북쪽 (오름차순) ✅")
                    else:
                        print(f"  {data_type}: 북쪽→남쪽 (내림차순) - 수정됨")
            except Exception as e:
                print(f"  {data_type}: 좌표 확인 오류 - {e}")
    
    def __len__(self):
        return len(self.paths)
    
    def __getitem__(self, idx):
        low_f, high_f = self.paths[idx]
        
        # 좌표 방향 확인하여 로딩
        low_arr = load_nc_with_coord_check(low_f, 'precipitation', (LAT_VALS.size, LON_VALS.size))
        high_arr = load_nc_with_coord_check(high_f, 'precipitation')
        
        # 정규화
        low_norm = (low_arr - self.stats['low_res']['mean']) / (self.stats['low_res']['std'] + 1e-8)
        high_norm = (high_arr - self.stats['high_res']['mean']) / (self.stats['high_res']['std'] + 1e-8)
        dem_norm = (self.dem - self.stats['dem']['mean']) / (self.stats['dem']['std'] + 1e-8)
        
        x = torch.tensor(np.stack([low_norm, dem_norm], axis=0), dtype=torch.float32)
        y = torch.tensor(high_norm, dtype=torch.float32).unsqueeze(0)
        return x, y
    
    def _calc_stats(self):
        print(f"\n=== 정규화 통계 계산 ===")
        lows, highs = [], []
        
        for i, (l, h) in enumerate(self.paths):
            if i % 100 == 0:
                print(f"  통계 계산 중: {i}/{len(self.paths)}")
            
            # 좌표 방향 수정하여 로딩
            low_data = load_nc_with_coord_check(l, 'precipitation', (LAT_VALS.size, LON_VALS.size))
            high_data = load_nc_with_coord_check(h, 'precipitation')
            
            lows.append(low_data)
            highs.append(high_data)
        
        lows = np.stack(lows)
        highs = np.stack(highs)
        
        stats = {
            'low_res':  {'mean': lows.mean(),  'std': lows.std()},
            'high_res': {'mean': highs.mean(), 'std': highs.std()},
            'dem':      {'mean': self.dem.mean(), 'std': self.dem.std()}
        }
        
        print("✅ 정규화 통계 완료:")
        for k, v in stats.items():
            print(f"  {k}: mean={v['mean']:.4f}, std={v['std']:.4f}")
            
        return stats

# ─────────────────────────────────────────────────────────
# 5) Generator 모델 정의
# ─────────────────────────────────────────────────────────
class Generator(nn.Module):
    def __init__(self, in_ch=2, n_layers=4, base_filters=32):
        super().__init__()
        layers = []
        c = in_ch
        for i in range(n_layers):
            f = base_filters * (2**i)
            layers += [nn.Conv2d(c,f,3,1,1,bias=False), nn.BatchNorm2d(f), nn.ReLU(True)]
            c = f
        self.encoder = nn.Sequential(*layers)
        self.final = nn.Conv2d(c,1,3,1,1)
    
    def forward(self, x):
        return self.final(self.encoder(x))

# ─────────────────────────────────────────────────────────
# 6) 학습 과정 표시 개선된 학습 함수
# ─────────────────────────────────────────────────────────
def train_final(train_ds, val_ds, test_ds, params, log_dir, epochs=50):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    print(f"\n{'='*60}")
    print(f"🚀 모델 학습 시작")
    print(f"{'='*60}")
    print(f"디바이스: {device}")
    print(f"에포크 수: {epochs}")
    print(f"배치 크기: {params['batch_size']}")
    print(f"학습률: {params['lr_g']:.2e}")
    print(f"손실 함수: {params['loss_function']}")
    
    logging.info(f"Using device: {device}")
    
    # 모델 초기화
    G = Generator(2, params['n_layers_g'], params['n_filters_base_g']).to(device)
    optimG = optim.AdamW(G.parameters(), lr=params['lr_g'], betas=(0.5,0.999))
    criterion = nn.L1Loss() if params['loss_function']=='L1' else nn.MSELoss()
    scaler = GradScaler(enabled=torch.cuda.is_available())
    
    # 데이터로더
    tl = DataLoader(train_ds, batch_size=params['batch_size'], shuffle=True, 
                   pin_memory=PIN_MEMORY, num_workers=NUM_WORKERS)
    vl = DataLoader(val_ds, batch_size=params['batch_size'], shuffle=False,
                   pin_memory=PIN_MEMORY, num_workers=NUM_WORKERS)
    
    print(f"배치 수 - Train: {len(tl)}, Val: {len(vl)}")
    
    # 메모리 및 속도 최적화 설정
    print(f"\n🚀 최적화 설정:")
    print(f"  Mixed Precision: {'✅ Enabled' if torch.cuda.is_available() else '❌ Disabled'}")
    print(f"  Data Loading: {NUM_WORKERS} workers, Pin Memory: {PIN_MEMORY}")
    
    # 학습률 스케줄러 추가 (선택사항)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimG, mode='min', factor=0.5, patience=5)
    
    best, cnt = 1e9, 0
    train_losses, val_losses = [], []
    
    print(f"\n{'='*70}")
    print(f"{'Epoch':>5} {'Train Loss':>12} {'Val Loss':>10} {'Best Val':>10} {'Status':>12} {'Time':>8}")
    print(f"{'='*70}")
    
    import time
    total_start_time = time.time()
    
    for ep in range(1, epochs+1):
        epoch_start_time = time.time()
        
        # 학습 모드
        G.train()
        train_loss = 0
        batch_count = 0
        
        # 에포크별 진행바 (더 간단하게)
        for batch_idx, (x, y) in enumerate(tl):
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            optimG.zero_grad()
            
            with autocast(device_type=device.type, enabled=torch.cuda.is_available()):
                loss = criterion(G(x), y) * params['content_weight']
            
            scaler.scale(loss).backward()
            scaler.step(optimG)
            scaler.update()
            train_loss += loss.item()
            batch_count += 1
            
            # 진행 상황 출력 (10배치마다)
            if batch_idx % 10 == 0:
                print(f"\r  Epoch {ep:2d} - Batch {batch_idx:3d}/{len(tl)} - Loss: {loss.item():.4f}", end='', flush=True)
        
        train_loss /= max(batch_count, 1)
        train_losses.append(train_loss)
        
        # 검증 모드 (빠른 검증)
        G.eval()
        val_loss = 0
        val_batch_count = 0
        
        with torch.no_grad():
            for x, y in vl:
                x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
                val_loss += criterion(G(x), y).item()
                val_batch_count += 1
        
        val_loss /= max(val_batch_count, 1)
        val_losses.append(val_loss)
        
        # 학습률 스케줄러 업데이트
        scheduler.step(val_loss)
        
        # 에포크 시간 계산
        epoch_time = time.time() - epoch_start_time
        
        # 결과 출력
        status = ""
        if val_loss < best:
            best, cnt = val_loss, 0
            torch.save(G.state_dict(), os.path.join(log_dir,'best_generator.pth'))
            status = "💾 SAVED"
        else:
            cnt += 1
            status = f"⏳ {cnt}/15"
        
        print(f"\r{ep:5d} {train_loss:12.6f} {val_loss:10.6f} {best:10.6f} {status:>12} {epoch_time:6.1f}s")
        
        # 로그 기록
        logging.info(f"Epoch {ep}, Train {train_loss:.6f}, Val {val_loss:.6f}, Time: {epoch_time:.1f}s")
        
        # Early stopping (더 aggressive)
        if cnt >= 10:  # 15 -> 10으로 줄임
            print(f"\n⏹️  Early stopping at epoch {ep} (no improvement for 10 epochs)")
            break
        
        # 3 에포크마다 진행 상황 요약 (5 -> 3으로 줄임)
        if ep % 3 == 0:
            recent_train = np.mean(train_losses[-3:])
            recent_val = np.mean(val_losses[-3:])
            elapsed_time = time.time() - total_start_time
            print(f"    📊 최근 3 에포크 평균 - Train: {recent_train:.6f}, Val: {recent_val:.6f} | 경과: {elapsed_time:.1f}s")
    
    print(f"{'='*70}")
    print(f"🎯 학습 완료! 최고 검증 손실: {best:.6f}")
    
    # 테스트 평가
    print(f"\n=== 테스트 평가 ===")
    G.load_state_dict(torch.load(os.path.join(log_dir,'best_generator.pth')))
    tle = DataLoader(test_ds, batch_size=1, shuffle=False)
    ms, ma = [], []
    
    G.eval()
    with torch.no_grad():
        for i, (x, y) in enumerate(tqdm(tle, desc="Testing", ncols=80)):
            x, y = x.to(device), y.to(device)
            p = G(x)
            y_, p_ = y.cpu().numpy().flatten(), p.cpu().numpy().flatten()
            ms.append(mean_squared_error(y_, p_))
            ma.append(mean_absolute_error(y_, p_))
            
            # 첫 5개 샘플 통계 출력
            if i < 5:
                print(f"  샘플 {i+1}: MSE={ms[-1]:.6f}, MAE={ma[-1]:.6f}")
    
    final_mse, final_mae = np.mean(ms), np.mean(ma)
    print(f"\n📈 최종 테스트 결과:")
    print(f"  평균 MSE: {final_mse:.6f}")
    print(f"  평균 MAE: {final_mae:.6f}")
    
    return G, final_mse, final_mae

# ─────────────────────────────────────────────────────────
# 7) 좌표 수정된 추론 함수
# ─────────────────────────────────────────────────────────
def run_inference(model, train_ds, low_res_dir, output_dir, start_date, end_date):
    print(f"\n{'='*60}")
    print(f"🔮 추론 및 NetCDF 생성")
    print(f"{'='*60}")
    
    device = next(model.parameters()).device
    model.eval()
    dem_norm = (train_ds.dem - train_ds.stats['dem']['mean']) / (train_ds.stats['dem']['std'] + 1e-8)
    dates = [start_date + datetime.timedelta(days=i) for i in range((end_date-start_date).days+1)]
    
    print(f"처리할 날짜 수: {len(dates)}일")
    print(f"출력 디렉토리: {output_dir}")
    
    processed_count = 0
    
    for date in tqdm(dates, desc="Downscaling", ncols=80):
        lf = os.path.join(low_res_dir, date.strftime('%Y%m%d') + '.nc4')
        if not os.path.exists(lf):
            continue
        
        try:
            # 좌표 방향 수정하여 GPM 로딩
            arr = load_nc_with_coord_check(lf, 'precipitation', (LAT_VALS.size, LON_VALS.size))
            
            # 정규화
            low_norm = (arr - train_ds.stats['low_res']['mean']) / (train_ds.stats['low_res']['std'] + 1e-8)
            
            # 예측
            x = torch.tensor(np.stack([low_norm, dem_norm], axis=0), dtype=torch.float32).unsqueeze(0).to(device)
            with torch.no_grad():
                pred = model(x).cpu().squeeze().numpy()
            
            # 역정규화
            pred_den = pred * train_ds.stats['high_res']['std'] + train_ds.stats['high_res']['mean']
            pred_den = np.clip(pred_den, 0, None)
            
            # NetCDF 저장
            save_to_netcdf(pred_den, date.strftime('%Y%m%d'), output_dir)
            processed_count += 1
            
            # 진행 상황 출력
            if processed_count % 100 == 0:
                print(f"  ✅ {processed_count}개 파일 처리 완료")
                
        except Exception as e:
            print(f"  ❌ 오류 ({date.strftime('%Y%m%d')}): {e}")
            continue
    
    print(f"\n🎉 추론 완료!")
    print(f"  총 처리된 파일: {processed_count}개")

# ─────────────────────────────────────────────────────────
# 8) 메인 실행
# ─────────────────────────────────────────────────────────
def main():
    print("🌧️ 강수량 다운스케일링 (GeoTIFF DEM 버전)")
    print(f"실행 시간: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    
    random.seed(42)
    np.random.seed(42)
    torch.manual_seed(42)
    
    # 로그 디렉토리
    run_dt = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
    log_dir = f"/mnt/c/Yeonji/2025.01.Drought/2004/{run_dt}"
    os.makedirs(log_dir, exist_ok=True)
    
    logging.basicConfig(level=logging.INFO, filename=os.path.join(log_dir,'train.log'),
                       format='%(asctime)s - %(message)s')
    
    # 디바이스 정보
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"사용 디바이스: {device}")
    
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
        print(f"GPU 메모리: {torch.cuda.get_device_properties(0).total_memory // 1024**3} GB")
    
    # 경로 설정
    low_res_dir = r"/mnt/c/Yeonji/GPM/Input"
    high_res_dir = r"/mnt/c/Yeonji/2025.01.Drought/2004/1.CoKriging(Daily)"
    dem_file = r"/mnt/c/Yeonji/2025.01.Drought/DEM(merge).tif"  # GeoTIFF
    
    print(f"\n📂 경로 확인:")
    print(f"  GPM 입력: {low_res_dir}")
    print(f"  Co-kriging: {high_res_dir}")
    print(f"  DEM 파일: {dem_file}")
    print(f"  로그 디렉토리: {log_dir}")
    
    # 경로 존재 확인
    for path_name, path in [("GPM", low_res_dir), ("Co-kriging", high_res_dir)]:
        if not os.path.exists(path):
            raise FileNotFoundError(f"{path_name} 디렉토리가 존재하지 않습니다: {path}")
        else:
            print(f"  ✅ {path_name} 디렉토리 확인됨")
    
    if not os.path.exists(dem_file):
        raise FileNotFoundError(f"DEM 파일이 존재하지 않습니다: {dem_file}")
    else:
        print(f"  ✅ DEM 파일 확인됨")
    
    print(f"\n{'='*60}")
    print(f"🔧 데이터셋 로딩 및 좌표 방향 수정")
    print(f"{'='*60}")
    
    # 데이터셋 생성
    train_ds = RainfallDataset(low_res_dir, high_res_dir, dem_file,
                              datetime.date(2004,1,1), datetime.date(2018,12,31))
    val_ds = RainfallDataset(low_res_dir, high_res_dir, dem_file,
                            datetime.date(2019,1,1), datetime.date(2021,12,31),
                            normalization_stats=train_ds.stats)
    test_ds = RainfallDataset(low_res_dir, high_res_dir, dem_file,
                             datetime.date(2022,1,1), datetime.date(2023,12,31),
                             normalization_stats=train_ds.stats)
    
    print(f"\n📊 데이터셋 크기:")
    print(f"  Train: {len(train_ds):,}개")
    print(f"  Validation: {len(val_ds):,}개")
    print(f"  Test: {len(test_ds):,}개")
    print(f"  총합: {len(train_ds) + len(val_ds) + len(test_ds):,}개")
    
    # 하이퍼파라미터 (속도 최적화)
    best_params = {
        'batch_size': 64,  # 32 -> 64로 증가 (GPU 메모리 허용 시)
        'lr_g': 3.0e-05,   # 학습률 약간 증가로 빠른 수렴
        'content_weight': 137.9359,
        'n_layers_g': 3,   # 4 -> 3으로 줄여서 모델 경량화
        'n_filters_base_g': 32,
        'loss_function': 'L1'
    }
    
    print(f"\n⚙️ 하이퍼파라미터:")
    for key, value in best_params.items():
        if isinstance(value, float):
            print(f"  {key}: {value:.2e}" if value < 0.001 else f"  {key}: {value:.4f}")
        else:
            print(f"  {key}: {value}")
    
    print(f"\n{'='*60}")
    print(f"🚀 모델 학습")
    print(f"{'='*60}")
    
    # 모델 학습 (에포크 수 줄임)
    model, mse, mae = train_final(train_ds, val_ds, test_ds, best_params, log_dir, epochs=30)  # 50 -> 30
    
    print(f"\n🎯 최종 성능:")
    print(f"  Test MSE: {mse:.6f}")
    print(f"  Test MAE: {mae:.6f}")
    
    # 모델 정보 저장
    model_info_file = os.path.join(log_dir, 'model_info.txt')
    with open(model_info_file, 'w') as f:
        f.write(f"모델 학습 완료 시간: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"디바이스: {device}\n")
        f.write(f"데이터셋 크기: Train={len(train_ds)}, Val={len(val_ds)}, Test={len(test_ds)}\n")
        f.write(f"최종 성능: MSE={mse:.6f}, MAE={mae:.6f}\n")
        f.write(f"DEM 파일: {dem_file} (GeoTIFF)\n")
        f.write(f"좌표 수정: Y축 방향 통일 (남쪽→북쪽)\n")
        f.write(f"하이퍼파라미터:\n")
        for k, v in best_params.items():
            f.write(f"  {k}: {v}\n")
        f.write(f"좌표 설정:\n")
        f.write(f"  위도 범위: {JEJU_LAT_START}~{JEJU_LAT_END} (남쪽→북쪽)\n")
        f.write(f"  경도 범위: {JEJU_LON_START}~{JEJU_LON_END}\n")
        f.write(f"  격자 크기: {LAT_VALS.size}×{LON_VALS.size}\n")
    
    print(f"📋 모델 정보가 저장되었습니다: {model_info_file}")
    
    # 추론
    print(f"\n{'='*60}")
    print(f"🔮 추론 및 NetCDF 생성")
    print(f"{'='*60}")
    
    out_folder = r"/mnt/c/Yeonji/2025.01.Drought/2004/0713(4)"
    run_inference(model, train_ds, low_res_dir, out_folder,
                  datetime.date(2004,1,1), datetime.date(2023,12,31))
    
    print(f"\n{'='*60}")
    print(f"🎉 전체 파이프라인 완료!")
    print(f"{'='*60}")
    print(f"📁 결과 파일:")
    print(f"  모델: {os.path.join(log_dir, 'best_generator.pth')}")
    print(f"  로그: {os.path.join(log_dir, 'train.log')}")
    print(f"  모델 정보: {model_info_file}")
    print(f"  다운스케일링 결과: {out_folder}")
    print(f"⏰ 완료 시간: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    
    # 결과 검증
    print(f"\n🔍 결과 검증:")
    try:
        # 생성된 파일 수 확인
        nc_files = [f for f in os.listdir(out_folder) if f.endswith('.nc')]
        print(f"  생성된 NetCDF 파일 수: {len(nc_files)}개")
        
        if nc_files:
            # 첫 번째 파일 상세 확인
            sample_file = os.path.join(out_folder, nc_files[0])
            with ncDataset(sample_file, 'r') as nc:
                lat = nc.variables['lat'][:]
                lon = nc.variables['lon'][:]
                precip = nc.variables['precipitation'][:]
                
                print(f"  샘플 파일: {nc_files[0]}")
                print(f"    위도 범위: {lat[0]:.3f}~{lat[-1]:.3f} ({'오름차순 ✅' if lat[0] < lat[-1] else '내림차순 ⚠️'})")
                print(f"    경도 범위: {lon[0]:.3f}~{lon[-1]:.3f}")
                print(f"    강수량 범위: {precip.min():.3f}~{precip.max():.3f} mm/day")
                print(f"    데이터 크기: {precip.shape}")
                print(f"    양수 강수량 비율: {(precip > 0).sum() / precip.size * 100:.1f}%")
                
                # 좌표 방향 최종 확인
                if lat[0] < lat[-1]:
                    print(f"  ✅ 좌표 방향 정상: 남쪽→북쪽 (DEM과 일치)")
                else:
                    print(f"  ⚠️  좌표 방향 확인 필요: 북쪽→남쪽")
                    
                # 좌표 정확성 확인
                expected_lat_start = JEJU_LAT_START
                expected_lat_end = JEJU_LAT_END
                if abs(lat[0] - expected_lat_start) < 0.01 and abs(lat[-1] - expected_lat_end) < 0.01:
                    print(f"  ✅ 좌표 정확성: 예상 범위와 일치")
                else:
                    print(f"  ⚠️  좌표 정확성: 예상({expected_lat_start:.3f}~{expected_lat_end:.3f})과 다름")
        
        # 마지막 몇 개 파일도 확인
        if len(nc_files) > 1:
            last_file = os.path.join(out_folder, nc_files[-1])
            with ncDataset(last_file, 'r') as nc:
                last_precip = nc.variables['precipitation'][:]
                print(f"  마지막 파일: {nc_files[-1]}")
                print(f"    강수량 범위: {last_precip.min():.3f}~{last_precip.max():.3f} mm/day")
                
    except Exception as e:
        print(f"  ❌ 검증 중 오류: {e}")
    
    # 성능 요약
    print(f"\n📈 최종 성능 요약:")
    print(f"  🎯 Test MSE: {mse:.6f}")
    print(f"  🎯 Test MAE: {mae:.6f}")
    print(f"  📊 처리된 기간: 2004-2023 (20년)")
    print(f"  🗺️  해상도: 0.1° → 0.01° (10배 향상)")
    print(f"  📐 격자 크기: {LAT_VALS.size} × {LON_VALS.size}")
    print(f"  🏔️  DEM: GeoTIFF 형식, 좌표 방향 자동 수정")
    
    print(f"\n{'='*60}")
    print(f"🎊 모든 작업이 성공적으로 완료되었습니다!")
    print(f"{'='*60}")

if __name__ == '__main__':
    try:
        main()
    except Exception as e:
        print(f"\n❌ 실행 중 오류 발생: {e}")
        logging.error(f"메인 함수 실행 중 오류: {e}", exc_info=True)
        print(f"자세한 오류 정보는 로그 파일을 확인하세요.")
        raise